# A06 · Customer behavior — dữ liệu đến RNN

Notebook là nơi đọc và sửa toàn bộ implementation dưới đây: dữ liệu, mô hình, loss, optimizer, train/đánh giá. Không cần mở file `.py` để xem thuật toán. Các bản `.py` cũ giữ nguyên để bảo toàn backend/artifact.

**Cách đọc:** khởi tạo → code từng hàm/lớp → thực hành theo dataset → kết quả. `%%inline_module` chỉ chạy code có ngay trong cell ở namespace riêng, không nạp implementation từ `.py`. Mỗi framework dùng kernel WSL riêng; code của cả hai vẫn hiện trong notebook. Mặc định không train/TEST lại.

In [1]:
PROJECT_NAME = 'A06'
from pathlib import Path
import sys, os, types, __future__
from IPython.core.magic import register_cell_magic
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ASSIGN_REORGANIZATION_EXECUTION_PLAN.md").exists())
sys.path.insert(0, str(REPO / "_shared"))
from assignment_runtime import activate
ACTIVE_FRAMEWORK = "keras" if ".venv-keras" in sys.executable else "torch"
_context = activate(PROJECT_NAME, ACTIVE_FRAMEWORK)
LOGICAL_ROOT = Path(_context["logical_root"])
RUN_EXPERIMENTS = False
RUN_TRAINING = False
RUN_TEST = False
# Kết quả của lượt chạy mới nằm ngoài notebooks/, không ghi vào artifact gốc.
import assignment_runtime as _paths
_paths._WRITE_CONTEXT["output"] = REPO / PROJECT_NAME / "artifacts/notebook_runs/inline_v1"
_paths._WRITE_CONTEXT["overlay"] = False
INLINE_MODULES = {}

def _notebook_package(name):
    if not name:
        return None
    if name not in INLINE_MODULES:
        parent, _, child = name.rpartition(".")
        package = _notebook_package(parent)
        module = types.ModuleType(name)
        module.__package__ = parent
        module.__path__ = []  # Không tìm implementation khác trong file .py.
        module.__file__ = str(LOGICAL_ROOT / (name.replace(".", "/") + "/__init__.py"))
        module.__dict__.update(ACTIVE_FRAMEWORK=ACTIVE_FRAMEWORK, RUN_TRAINING=False, RUN_TEST=False)
        INLINE_MODULES[name] = module
        sys.modules[name] = module
        if package is not None:
            setattr(package, child, module)
    return INLINE_MODULES[name]

@register_cell_magic
def inline_module(line, cell):
    """Chạy đúng code trong cell ở namespace riêng; không đọc code từ .py."""
    name, relative, scope = line.split()
    if scope != "both" and scope != ACTIVE_FRAMEWORK:
        return
    parent, _, child = name.rpartition(".")
    _notebook_package(parent)
    module = _notebook_package(name)
    module.__file__ = str(LOGICAL_ROOT / relative)
    module.__package__ = name if relative.endswith("/__init__.py") else parent
    compiled = compile(cell, "<notebook>/" + name, "exec", flags=__future__.annotations.compiler_flag, dont_inherit=True)
    exec(compiled, module.__dict__)
    module.__notebook_source__ = True

print({"bài": PROJECT_NAME, "kernel": ACTIVE_FRAMEWORK, "train": RUN_TRAINING, "TEST": RUN_TEST})


{'bài': 'A06', 'kernel': 'keras', 'train': False, 'TEST': False}


### `src.config`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [2]:
%%inline_module src.config A06/src/config.py both
"""Single source of truth for deterministic A06 preprocessing."""

from __future__ import annotations

from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[1]

SEED = 42

SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}

RAW_CUSTOMER_PATH = PROJECT_ROOT / "datasets" / "customer" / "online_retail_II.xlsx"

RAW_STOCK_PATH = PROJECT_ROOT / "datasets" / "stock" / "AAPL_2015_2025.csv"

PROCESSED_DIR = PROJECT_ROOT / "datasets" / "processed"

PREPROCESSING_MODEL_DIR = PROJECT_ROOT / "models" / "preprocessing"

METRICS_DIR = PROJECT_ROOT / "results" / "metrics"

PREPROCESSING_METADATA_PATH = METRICS_DIR / "preprocessing_metadata.json"

EXPECTED_CUSTOMER_SHA256 = (
    "bcbe73b35f5b7babf197fb0cb983a11f5d9ff929078d4aa53d171b1f2df2e980"
)

EXPECTED_STOCK_SHA256 = (
    "2c8eef3c6a3836e2217bf3e99faaeec37d30e60298f7319e8f3748b71e5bd2b8"
)

CUSTOMER_COLUMN_ALIASES = {
    "Invoice": "InvoiceNo",
    "Price": "UnitPrice",
    "Customer ID": "CustomerID",
}

CUSTOMER_REQUIRED_COLUMNS = (
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
)

CUSTOMER_FEATURES = (
    "total_spent",
    "total_quantity",
    "order_count",
    "unique_products",
    "active_flag",
)

CUSTOMER_SEQUENCE_LENGTH = 8

CUSTOMER_WEEK_FREQUENCY = "W-MON"

STOCK_REQUIRED_COLUMNS = (
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Adj Close",
    "Volume",
)

STOCK_FEATURES = ("Open", "High", "Low", "Close", "Volume")

STOCK_SEQUENCE_LENGTH = 30



#### ensure_output_directories

In [3]:
%%inline_module src.config A06/src/config.py both
def ensure_output_directories() -> None:
    """Create only the directories owned by the preprocessing milestone."""

    for directory in (PROCESSED_DIR, PREPROCESSING_MODEL_DIR, METRICS_DIR):
        directory.mkdir(parents=True, exist_ok=True)



### `src.preprocessing_utils`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [4]:
%%inline_module src.preprocessing_utils A06/src/preprocessing_utils.py both
"""Shared, framework-neutral preprocessing utilities."""

from __future__ import annotations

import hashlib

import json

import os

from pathlib import Path

from typing import Any, Mapping

import joblib

import numpy as np

import pandas as pd

from sklearn.preprocessing import StandardScaler

from src.config import SPLIT_RATIOS



#### sha256_file

In [5]:
%%inline_module src.preprocessing_utils A06/src/preprocessing_utils.py both
def sha256_file(path: Path) -> str:
    """Return a streaming SHA-256 digest without modifying ``path``."""

    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()



#### chronological_split_indices

In [6]:
%%inline_module src.preprocessing_utils A06/src/preprocessing_utils.py both
def chronological_split_indices(
    timestamps: np.ndarray | pd.Series | pd.Index,
    ratios: Mapping[str, float] = SPLIT_RATIOS,
) -> dict[str, np.ndarray]:
    """Split by timestamp groups while approximating sample-count ratios.

    All samples sharing a target timestamp remain in the same split. Boundaries
    are chosen deterministically at the cumulative sample counts nearest 70%
    and 85%, subject to leaving at least one distinct timestamp per split.
    """

    values = pd.to_datetime(np.asarray(timestamps), errors="coerce").to_numpy(
        dtype="datetime64[ns]"
    )
    if values.ndim != 1 or len(values) == 0:
        raise ValueError("timestamps must be a non-empty one-dimensional array")
    if np.isnat(values).any():
        raise ValueError("timestamps contain NaT")

    ratio_values = np.array([ratios[name] for name in ("train", "val", "test")])
    if np.any(ratio_values <= 0) or not np.isclose(ratio_values.sum(), 1.0):
        raise ValueError("train/val/test ratios must be positive and sum to 1")

    unique_times, counts = np.unique(values, return_counts=True)
    if len(unique_times) < 3:
        raise ValueError("at least three distinct target timestamps are required")

    cumulative = np.cumsum(counts)
    train_candidates = np.arange(0, len(unique_times) - 2)
    train_goal = ratios["train"] * len(values)
    train_end_position = train_candidates[
        np.argmin(np.abs(cumulative[train_candidates] - train_goal))
    ]

    validation_candidates = np.arange(train_end_position + 1, len(unique_times) - 1)
    validation_goal = (ratios["train"] + ratios["val"]) * len(values)
    validation_end_position = validation_candidates[
        np.argmin(np.abs(cumulative[validation_candidates] - validation_goal))
    ]

    train_cutoff = unique_times[train_end_position]
    validation_cutoff = unique_times[validation_end_position]
    result = {
        "train": np.flatnonzero(values <= train_cutoff),
        "val": np.flatnonzero((values > train_cutoff) & (values <= validation_cutoff)),
        "test": np.flatnonzero(values > validation_cutoff),
    }
    if any(len(indices) == 0 for indices in result.values()):
        raise AssertionError("chronological split unexpectedly produced an empty split")
    return result



#### fit_feature_scaler

In [7]:
%%inline_module src.preprocessing_utils A06/src/preprocessing_utils.py both
def fit_feature_scaler(train_x: np.ndarray) -> StandardScaler:
    """Fit a StandardScaler only on flattened TRAIN sequence observations."""

    train_array = np.asarray(train_x)
    if train_array.ndim != 3 or train_array.shape[-1] == 0:
        raise ValueError("train_x must have shape (samples, sequence_length, features)")
    if len(train_array) == 0 or not np.isfinite(train_array).all():
        raise ValueError("train_x must be non-empty and finite")
    scaler = StandardScaler()
    scaler.fit(train_array.reshape(-1, train_array.shape[-1]))
    return scaler



#### transform_sequences

In [8]:
%%inline_module src.preprocessing_utils A06/src/preprocessing_utils.py both
def transform_sequences(scaler: StandardScaler, sequences: np.ndarray) -> np.ndarray:
    """Apply a fitted feature scaler and restore the 3-D sequence shape."""

    values = np.asarray(sequences)
    if values.ndim != 3:
        raise ValueError("sequences must be three-dimensional")
    transformed = scaler.transform(values.reshape(-1, values.shape[-1]))
    return transformed.reshape(values.shape).astype(np.float32, copy=False)



#### save_npz_atomic

In [9]:
%%inline_module src.preprocessing_utils A06/src/preprocessing_utils.py both
def save_npz_atomic(path: Path, **arrays: np.ndarray) -> None:
    """Write a compressed NPZ then atomically replace the destination."""

    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f".{destination.name}.tmp")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, destination)



#### save_joblib_atomic

In [10]:
%%inline_module src.preprocessing_utils A06/src/preprocessing_utils.py both
def save_joblib_atomic(path: Path, value: Any) -> None:
    """Persist a Python object without leaving a partial final file."""

    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f".{destination.name}.tmp")
    joblib.dump(value, temporary)
    os.replace(temporary, destination)



#### write_json_atomic

In [11]:
%%inline_module src.preprocessing_utils A06/src/preprocessing_utils.py both
def write_json_atomic(path: Path, payload: Mapping[str, Any]) -> None:
    """Write sorted UTF-8 JSON with an atomic final replacement."""

    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f".{destination.name}.tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, destination)



#### artifact_record

In [12]:
%%inline_module src.preprocessing_utils A06/src/preprocessing_utils.py both
def artifact_record(path: Path, project_root: Path) -> dict[str, Any]:
    """Describe a saved artifact with a project-relative path and checksum."""

    artifact = Path(path)
    return {
        "path": artifact.relative_to(project_root).as_posix(),
        "bytes": artifact.stat().st_size,
        "sha256": sha256_file(artifact),
    }



#### iso_date

In [13]:
%%inline_module src.preprocessing_utils A06/src/preprocessing_utils.py both
def iso_date(value: np.datetime64 | pd.Timestamp) -> str:
    """Serialize a date-like value as YYYY-MM-DD."""

    return pd.Timestamp(value).date().isoformat()



### `src.customer_pipeline`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [14]:
%%inline_module src.customer_pipeline A06/src/customer_pipeline.py both
"""Deterministic customer-week preprocessing shared by both frameworks."""

from __future__ import annotations

from pathlib import Path

from typing import Any

import numpy as np

import pandas as pd

from src.config import (
    CUSTOMER_COLUMN_ALIASES,
    CUSTOMER_FEATURES,
    CUSTOMER_REQUIRED_COLUMNS,
    CUSTOMER_SEQUENCE_LENGTH,
    CUSTOMER_WEEK_FREQUENCY,
    EXPECTED_CUSTOMER_SHA256,
    PREPROCESSING_MODEL_DIR,
    PROCESSED_DIR,
    PROJECT_ROOT,
    RAW_CUSTOMER_PATH,
    SPLIT_RATIOS,
)

from src.preprocessing_utils import (
    artifact_record,
    chronological_split_indices,
    fit_feature_scaler,
    iso_date,
    save_joblib_atomic,
    save_npz_atomic,
    sha256_file,
    transform_sequences,
)



#### load_customer_workbook

In [15]:
%%inline_module src.customer_pipeline A06/src/customer_pipeline.py both
def load_customer_workbook(path: Path = RAW_CUSTOMER_PATH) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Read all raw sheets only after validating their actual schemas."""

    source = Path(path)
    frames: list[pd.DataFrame] = []
    sheet_info: list[dict[str, Any]] = []
    schemas: list[tuple[str, ...]] = []
    with pd.ExcelFile(source) as excel_file:
        sheet_names = list(excel_file.sheet_names)
        for sheet_name in sheet_names:
            frame = pd.read_excel(excel_file, sheet_name=sheet_name)
            frames.append(frame)
            schema = tuple(map(str, frame.columns))
            schemas.append(schema)
            sheet_info.append(
                {
                    "name": sheet_name,
                    "rows": int(len(frame)),
                    "columns": list(schema),
                    "dtypes": {
                        column: str(dtype) for column, dtype in frame.dtypes.items()
                    },
                }
            )

    if not frames:
        raise ValueError("customer workbook has no sheets")
    if any(schema != schemas[0] for schema in schemas[1:]):
        raise ValueError("customer workbook sheets have incompatible schemas")

    combined = pd.concat(frames, ignore_index=True)
    return combined, {"sheet_names": sheet_names, "sheets": sheet_info}



#### normalize_customer_columns

In [16]:
%%inline_module src.customer_pipeline A06/src/customer_pipeline.py both
def normalize_customer_columns(raw: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, str]]:
    """Normalize historical names in memory and validate the required schema."""

    rename_map = {
        source: target
        for source, target in CUSTOMER_COLUMN_ALIASES.items()
        if source in raw.columns
    }
    normalized = raw.rename(columns=rename_map).copy()
    if normalized.columns.duplicated().any():
        raise ValueError("column normalization produced duplicate column names")
    missing = sorted(set(CUSTOMER_REQUIRED_COLUMNS) - set(normalized.columns))
    if missing:
        raise ValueError(f"customer data is missing required columns: {missing}")
    return normalized, rename_map



#### clean_customer_transactions

In [17]:
%%inline_module src.customer_pipeline A06/src/customer_pipeline.py both
def clean_customer_transactions(raw: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """Apply the purchase-only cleaning rules confirmed in Notebook 02."""

    data, _ = normalize_customer_columns(raw)
    data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"], errors="coerce")
    for column in ("CustomerID", "Quantity", "UnitPrice"):
        data[column] = pd.to_numeric(data[column], errors="coerce")

    invoice_text = data["InvoiceNo"].astype("string").str.strip()
    cancellation = invoice_text.str.upper().str.startswith("C", na=False)
    customer_integer = data["CustomerID"].mod(1).eq(0).fillna(False)
    valid_customer = data["CustomerID"].notna() & data["CustomerID"].gt(0) & customer_integer

    duplicate_count = int(data.duplicated().sum())
    counts = {
        "raw_rows": int(len(data)),
        "exact_duplicates_removed": duplicate_count,
        "invalid_or_missing_customer_id_raw": int((~valid_customer).sum()),
        "invalid_invoice_date_raw": int(data["InvoiceDate"].isna().sum()),
        "cancellation_lines_raw": int(cancellation.sum()),
        "nonpositive_quantity_lines_raw": int(data["Quantity"].le(0).fillna(True).sum()),
        "nonpositive_unit_price_lines_raw": int(data["UnitPrice"].le(0).fillna(True).sum()),
    }

    deduplicated = data.drop_duplicates().copy()
    dedup_invoice_text = deduplicated["InvoiceNo"].astype("string").str.strip()
    dedup_cancellation = dedup_invoice_text.str.upper().str.startswith("C", na=False)
    dedup_integer_customer = deduplicated["CustomerID"].mod(1).eq(0).fillna(False)
    clean_mask = (
        deduplicated["CustomerID"].notna()
        & deduplicated["CustomerID"].gt(0)
        & dedup_integer_customer
        & deduplicated["InvoiceDate"].notna()
        & ~dedup_cancellation
        & deduplicated["Quantity"].gt(0)
        & deduplicated["UnitPrice"].gt(0)
    )

    clean = deduplicated.loc[clean_mask].copy()
    clean["InvoiceNo"] = clean["InvoiceNo"].astype("string").str.strip()
    clean["CustomerID"] = clean["CustomerID"].astype("int64")
    clean["Revenue"] = clean["Quantity"] * clean["UnitPrice"]
    if not clean["Revenue"].gt(0).all():
        raise AssertionError("cleaned customer Revenue must be strictly positive")

    counts.update(
        {
            "cleaned_rows": int(len(clean)),
            "cleaned_customers": int(clean["CustomerID"].nunique()),
            "cleaned_invoices": int(clean["InvoiceNo"].nunique()),
        }
    )
    return clean, counts



#### aggregate_customer_weeks

In [18]:
%%inline_module src.customer_pipeline A06/src/customer_pipeline.py both
def aggregate_customer_weeks(clean: pd.DataFrame) -> pd.DataFrame:
    """Aggregate valid purchase lines to active customer-week rows."""

    required = {"CustomerID", "InvoiceDate", "InvoiceNo", "StockCode", "Quantity", "Revenue"}
    missing = sorted(required - set(clean.columns))
    if missing:
        raise ValueError(f"clean customer data is missing columns: {missing}")

    data = clean.copy()
    data["WeekStart"] = data["InvoiceDate"].dt.to_period("W-SUN").dt.start_time
    weekly = (
        data.groupby(["CustomerID", "WeekStart"], observed=True, sort=True)
        .agg(
            total_spent=("Revenue", "sum"),
            total_quantity=("Quantity", "sum"),
            order_count=("InvoiceNo", "nunique"),
            unique_products=("StockCode", "nunique"),
        )
        .reset_index()
        .sort_values(["CustomerID", "WeekStart"], kind="stable")
        .reset_index(drop=True)
    )
    weekly["active_flag"] = np.int8(1)
    if weekly.duplicated(["CustomerID", "WeekStart"]).any():
        raise AssertionError("customer-week aggregation produced duplicate keys")
    if not weekly["order_count"].ge(1).all():
        raise AssertionError("active customer weeks must contain at least one unique invoice")
    return weekly



#### full_week_bounds

In [19]:
%%inline_module src.customer_pipeline A06/src/customer_pipeline.py both
def full_week_bounds(active_weekly: pd.DataFrame) -> tuple[pd.Timestamp, pd.Timestamp]:
    """Exclude the two partial boundary weeks identified in Notebook 02."""

    first_observed = pd.Timestamp(active_weekly["WeekStart"].min())
    last_observed = pd.Timestamp(active_weekly["WeekStart"].max())
    first_full = first_observed + pd.Timedelta(weeks=1)
    last_full = last_observed - pd.Timedelta(weeks=1)
    if first_full >= last_full:
        raise ValueError("customer observation period has too few complete weeks")
    return first_full, last_full



#### build_customer_sequences

In [20]:
%%inline_module src.customer_pipeline A06/src/customer_pipeline.py both
def build_customer_sequences(
    active_weekly: pd.DataFrame,
    *,
    sequence_length: int = CUSTOMER_SEQUENCE_LENGTH,
    first_full_week: pd.Timestamp,
    last_full_week: pd.Timestamp,
) -> dict[str, np.ndarray]:
    """Create regular zero-filled histories and next-week binary targets."""

    if sequence_length <= 0:
        raise ValueError("sequence_length must be positive")
    missing = sorted(
        {"CustomerID", "WeekStart", *CUSTOMER_FEATURES} - set(active_weekly.columns)
    )
    if missing:
        raise ValueError(f"active weekly data is missing columns: {missing}")

    weekly = active_weekly.copy()
    weekly["WeekStart"] = pd.to_datetime(weekly["WeekStart"], errors="raise")
    weekly = weekly.sort_values(["CustomerID", "WeekStart"], kind="stable")
    if weekly.duplicated(["CustomerID", "WeekStart"]).any():
        raise ValueError("active_weekly must contain one row per customer-week")
    if not weekly["active_flag"].eq(1).all():
        raise ValueError("active_weekly rows must all have active_flag=1")

    first_full = pd.Timestamp(first_full_week)
    last_full = pd.Timestamp(last_full_week)
    feature_arrays: list[np.ndarray] = []
    target_arrays: list[np.ndarray] = []
    customer_arrays: list[np.ndarray] = []
    week_arrays: list[np.ndarray] = []

    for customer_id, customer_rows in weekly.groupby("CustomerID", sort=True, observed=True):
        timeline_start = max(pd.Timestamp(customer_rows["WeekStart"].min()), first_full)
        if timeline_start > last_full:
            continue
        week_index = pd.date_range(timeline_start, last_full, freq=CUSTOMER_WEEK_FREQUENCY)
        if len(week_index) <= sequence_length:
            continue

        timeline = (
            customer_rows.set_index("WeekStart")
            .reindex(week_index)[list(CUSTOMER_FEATURES)]
            .fillna(0.0)
        )
        values = timeline.to_numpy(dtype=np.float32)
        windows = np.lib.stride_tricks.sliding_window_view(
            values, window_shape=sequence_length, axis=0
        )[:-1].transpose(0, 2, 1).copy()
        targets = values[sequence_length:, CUSTOMER_FEATURES.index("active_flag")].astype(
            np.int8
        )
        target_weeks = week_index[sequence_length:].to_numpy(dtype="datetime64[D]")

        if len(windows) != len(targets):
            raise AssertionError("customer windows and targets are misaligned")
        feature_arrays.append(windows)
        target_arrays.append(targets)
        customer_arrays.append(np.full(len(targets), int(customer_id), dtype=np.int64))
        week_arrays.append(target_weeks)

    if not feature_arrays:
        raise ValueError("no eligible customer sequences were generated")

    result = {
        "X": np.concatenate(feature_arrays),
        "y": np.concatenate(target_arrays),
        "customer_id": np.concatenate(customer_arrays),
        "target_week": np.concatenate(week_arrays),
    }
    if result["X"].shape[1:] != (sequence_length, len(CUSTOMER_FEATURES)):
        raise AssertionError("customer sequence shape does not match the contract")
    return result



#### _class_distribution

In [21]:
%%inline_module src.customer_pipeline A06/src/customer_pipeline.py both
def _class_distribution(labels: np.ndarray) -> dict[str, Any]:
    counts = np.bincount(labels.astype(np.int64), minlength=2)
    total = int(counts.sum())
    return {
        "class_0": int(counts[0]),
        "class_1": int(counts[1]),
        "class_0_rate": float(counts[0] / total),
        "class_1_rate": float(counts[1] / total),
    }



#### prepare_customer_data

In [22]:
%%inline_module src.customer_pipeline A06/src/customer_pipeline.py both
def prepare_customer_data(
    raw_path: Path = RAW_CUSTOMER_PATH,
    processed_dir: Path = PROCESSED_DIR,
    scaler_dir: Path = PREPROCESSING_MODEL_DIR,
) -> dict[str, Any]:
    """Run the complete customer preprocessing stage and save shared artifacts."""

    source = Path(raw_path)
    source_hash_before = sha256_file(source)
    if source_hash_before != EXPECTED_CUSTOMER_SHA256:
        raise ValueError("locked customer workbook checksum does not match PROJECT_SPEC state")

    raw, workbook_info = load_customer_workbook(source)
    normalized_preview, rename_map = normalize_customer_columns(raw.iloc[:0])
    del normalized_preview
    clean, cleaning_counts = clean_customer_transactions(raw)
    del raw
    active_weekly = aggregate_customer_weeks(clean)
    clean_date_min = pd.Timestamp(clean["InvoiceDate"].min())
    clean_date_max = pd.Timestamp(clean["InvoiceDate"].max())
    del clean

    first_full_week, last_full_week = full_week_bounds(active_weekly)
    sequences = build_customer_sequences(
        active_weekly,
        sequence_length=CUSTOMER_SEQUENCE_LENGTH,
        first_full_week=first_full_week,
        last_full_week=last_full_week,
    )
    active_week_rows = int(len(active_weekly))
    del active_weekly

    split_indices = chronological_split_indices(sequences["target_week"])
    raw_split_x: dict[str, np.ndarray] = {}
    split_payloads: dict[str, dict[str, np.ndarray]] = {}
    for split_name, indices in split_indices.items():
        order = np.lexsort(
            (sequences["customer_id"][indices], sequences["target_week"][indices])
        )
        ordered_indices = indices[order]
        raw_split_x[split_name] = sequences["X"][ordered_indices]
        split_payloads[split_name] = {
            "y": sequences["y"][ordered_indices],
            "customer_id": sequences["customer_id"][ordered_indices],
            "target_week": sequences["target_week"][ordered_indices],
        }
    del sequences

    feature_scaler = fit_feature_scaler(raw_split_x["train"])
    feature_scaler_path = Path(scaler_dir) / "customer_feature_scaler.joblib"
    save_joblib_atomic(feature_scaler_path, feature_scaler)

    artifact_paths: dict[str, Path] = {}
    for split_name in ("train", "val", "test"):
        artifact_path = Path(processed_dir) / f"customer_{split_name}.npz"
        payload = split_payloads[split_name]
        save_npz_atomic(
            artifact_path,
            X=transform_sequences(feature_scaler, raw_split_x[split_name]),
            y=payload["y"].astype(np.int8, copy=False),
            customer_id=payload["customer_id"].astype(np.int64, copy=False),
            target_week=payload["target_week"].astype("datetime64[D]", copy=False),
        )
        artifact_paths[split_name] = artifact_path

    train_distribution = _class_distribution(split_payloads["train"]["y"])
    if train_distribution["class_1"] == 0:
        raise ValueError("TRAIN customer split has no positive labels")
    positive_weight = train_distribution["class_0"] / train_distribution["class_1"]

    source_hash_after = sha256_file(source)
    if source_hash_after != source_hash_before:
        raise AssertionError("raw customer workbook changed during preprocessing")

    split_metadata: dict[str, Any] = {}
    for split_name in ("train", "val", "test"):
        payload = split_payloads[split_name]
        split_metadata[split_name] = {
            "X_shape": [
                int(len(payload["y"])),
                CUSTOMER_SEQUENCE_LENGTH,
                len(CUSTOMER_FEATURES),
            ],
            "y_shape": [int(len(payload["y"]))],
            "target_week_min": iso_date(payload["target_week"].min()),
            "target_week_max": iso_date(payload["target_week"].max()),
            "sample_ratio": float(
                len(payload["y"]) / sum(len(v["y"]) for v in split_payloads.values())
            ),
            "class_distribution": _class_distribution(payload["y"]),
        }

    return {
        "raw_source": {
            "path": source.relative_to(PROJECT_ROOT).as_posix(),
            "sha256_before": source_hash_before,
            "sha256_after": source_hash_after,
            "workbook": workbook_info,
        },
        "column_normalization": rename_map,
        "cleaning_rules": [
            "remove exact duplicate rows",
            "require positive integer CustomerID",
            "require valid InvoiceDate",
            "exclude InvoiceNo values beginning with C (case-insensitive)",
            "require Quantity > 0",
            "require UnitPrice > 0",
            "Revenue = Quantity * UnitPrice",
        ],
        "cleaning_counts": cleaning_counts,
        "cleaned_date_range": {
            "min": clean_date_min.isoformat(),
            "max": clean_date_max.isoformat(),
        },
        "weekly_definition": {
            "week": "Monday through Sunday; WeekStart is Monday",
            "total_spent": "sum of Revenue within customer-week",
            "total_quantity": "sum of positive Quantity within customer-week",
            "order_count": "number of unique InvoiceNo values within customer-week",
            "unique_products": "number of unique StockCode values within customer-week",
            "active_flag": "1 for at least one valid purchase, otherwise 0 after reindexing",
            "active_customer_week_rows": active_week_rows,
        },
        "eligibility_policy": {
            "first_full_week": iso_date(first_full_week),
            "last_full_week": iso_date(last_full_week),
            "timeline_start": "max(customer first purchase week, first_full_week)",
            "timeline_end": "last_full_week for every eligible customer",
            "inactive_week_fill": 0,
            "required_history_weeks": CUSTOMER_SEQUENCE_LENGTH,
        },
        "feature_names": list(CUSTOMER_FEATURES),
        "sequence_length": CUSTOMER_SEQUENCE_LENGTH,
        "target_definition": "active_flag of the immediately following week",
        "task": "binary classification",
        "split_strategy": {
            "basis": "target_week",
            "ratios_requested": dict(SPLIT_RATIOS),
            "boundary_rule": "nearest cumulative sample count without splitting a target_week",
            "context_rule": "validation/test inputs may use earlier known history",
        },
        "splits": split_metadata,
        "feature_scaler": {
            "type": "sklearn.preprocessing.StandardScaler",
            "fit_scope": "flattened TRAIN X only",
            "n_samples_seen": int(feature_scaler.n_samples_seen_),
            "mean": feature_scaler.mean_.tolist(),
            "scale": feature_scaler.scale_.tolist(),
            **artifact_record(feature_scaler_path, PROJECT_ROOT),
        },
        "class_weight": {
            "positive_weight": float(positive_weight),
            "formula": "TRAIN class_0 count / TRAIN class_1 count",
            "fit_scope": "TRAIN y only",
        },
        "artifacts": {
            name: artifact_record(path, PROJECT_ROOT) for name, path in artifact_paths.items()
        },
    }



### `src.keras_customer`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. Kernel keras.


#### Thiết lập / hằng số

In [23]:
%%inline_module src.keras_customer A06/src/keras_customer.py keras
"""Reusable Keras components for the A06 customer Vanilla RNN."""
from __future__ import annotations
from pathlib import Path
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
if ACTIVE_FRAMEWORK == 'keras':
    from tensorflow import keras
else:
    keras = None


I0000 00:00:1793493445.789999    1076 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793493445.831264    1076 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1793493449.443947    1076 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


#### validate_customer_arrays

In [24]:
%%inline_module src.keras_customer A06/src/keras_customer.py keras
def validate_customer_arrays(features: np.ndarray, labels: np.ndarray) -> None:
    """Validate the finalized customer sequence contract without changing data."""

    x = np.asarray(features)
    y = np.asarray(labels)
    if x.ndim != 3 or x.shape[1:] != (8, 5):
        raise ValueError("customer X must have shape (samples, 8, 5)")
    if y.ndim != 1 or len(x) != len(y):
        raise ValueError("customer y must be one-dimensional and align with X")
    if not np.isfinite(x).all() or not np.isfinite(y).all():
        raise ValueError("customer features and labels must be finite")
    if not np.isin(y, [0, 1]).all():
        raise ValueError("customer labels must be binary")



#### build_customer_rnn

In [25]:
%%inline_module src.keras_customer A06/src/keras_customer.py keras
def build_customer_rnn(
    *,
    input_shape: tuple[int, int] = (8, 5),
    hidden_units: int = 64,
    learning_rate: float = 1e-3,
) -> keras.Model:
    """Build and compile the Keras SimpleRNN binary-classification baseline."""

    if input_shape != (8, 5):
        raise ValueError("input_shape must remain (8, 5) for A06 customer data")
    if hidden_units <= 0 or learning_rate <= 0:
        raise ValueError("hidden_units and learning_rate must be positive")

    model = keras.Sequential(
        [
            keras.Input(shape=input_shape, name="customer_sequence"),
            keras.layers.SimpleRNN(hidden_units, activation="tanh", name="simple_rnn"),
            keras.layers.Dense(1, activation="sigmoid", name="purchase_probability"),
        ],
        name="keras_customer_vanilla_rnn",
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
    )
    return model



#### compute_class_weight

In [26]:
%%inline_module src.keras_customer A06/src/keras_customer.py keras
def compute_class_weight(train_labels: np.ndarray) -> dict[int, float]:
    """Return Keras class weights from the supplied TRAIN labels only."""

    labels = np.asarray(train_labels)
    negatives = int(np.count_nonzero(labels == 0))
    positives = int(np.count_nonzero(labels == 1))
    if negatives == 0 or positives == 0 or negatives + positives != len(labels):
        raise ValueError("train_labels must contain both binary classes and no other values")
    return {0: 1.0, 1: negatives / positives}



#### make_sample_weights

In [27]:
%%inline_module src.keras_customer A06/src/keras_customer.py keras
def make_sample_weights(
    labels: np.ndarray,
    class_weight: dict[int, float],
) -> np.ndarray:
    """Map binary labels to weights fixed from TRAIN without recomputing them."""

    y = np.asarray(labels)
    if y.ndim != 1 or not np.isin(y, [0, 1]).all():
        raise ValueError("labels must be a one-dimensional binary array")
    if set(class_weight) != {0, 1} or min(class_weight.values()) <= 0:
        raise ValueError("class_weight must provide positive weights for classes 0 and 1")
    return np.where(y == 1, class_weight[1], class_weight[0]).astype(np.float32)



#### build_callbacks

In [28]:
%%inline_module src.keras_customer A06/src/keras_customer.py keras
def build_callbacks(
    model_path: Path,
    *,
    patience: int = 5,
    min_delta: float = 1e-4,
) -> list[keras.callbacks.Callback]:
    """Create validation-loss callbacks that restore and persist the best model."""

    if patience <= 0 or min_delta < 0:
        raise ValueError("patience must be positive and min_delta non-negative")
    destination = Path(model_path)
    if destination.suffix != ".keras":
        raise ValueError("Keras model path must end with .keras")
    destination.parent.mkdir(parents=True, exist_ok=True)
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=patience,
            min_delta=min_delta,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=str(destination),
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            save_weights_only=False,
            verbose=0,
        ),
    ]



#### binary_classification_metrics

In [29]:
%%inline_module src.keras_customer A06/src/keras_customer.py keras
def binary_classification_metrics(
    labels: np.ndarray,
    probabilities: np.ndarray,
    *,
    threshold: float = 0.5,
) -> dict[str, float]:
    """Calculate fixed-threshold classification metrics and ROC-AUC."""

    y_true = np.asarray(labels).astype(np.int8)
    y_probability = np.asarray(probabilities, dtype=np.float64)
    if y_true.ndim != 1 or y_true.shape != y_probability.shape:
        raise ValueError("labels and probabilities must be aligned one-dimensional arrays")
    if not 0 <= threshold <= 1:
        raise ValueError("threshold must lie in [0, 1]")
    predictions = (y_probability >= threshold).astype(np.int8)
    return {
        "accuracy": float(accuracy_score(y_true, predictions)),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_probability)),
    }



### `src.pytorch_customer`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. Kernel torch.


#### Thiết lập / hằng số

In [30]:
%%inline_module src.pytorch_customer A06/src/pytorch_customer.py torch
"""Reusable PyTorch components for the A06 customer Vanilla RNN."""
from __future__ import annotations
import os
import time
from pathlib import Path
from typing import Any
import numpy as np
if ACTIVE_FRAMEWORK == 'torch':
    import torch
else:
    torch = None
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
if ACTIVE_FRAMEWORK == 'torch':
    from torch import nn
else:
    nn = None
from torch.utils.data import DataLoader, Dataset


#### CustomerSequenceDataset

In [31]:
%%inline_module src.pytorch_customer A06/src/pytorch_customer.py torch
class CustomerSequenceDataset(Dataset):
    """Expose preprocessed customer sequences without changing their values."""

    def __init__(self, features: np.ndarray, labels: np.ndarray) -> None:
        x = np.asarray(features)
        y = np.asarray(labels)
        if x.ndim != 3 or x.shape[1:] != (8, 5):
            raise ValueError("customer X must have shape (samples, 8, 5)")
        if y.ndim != 1 or len(x) != len(y):
            raise ValueError("customer y must be one-dimensional and align with X")
        if not np.isin(y, [0, 1]).all():
            raise ValueError("customer labels must be binary")
        self.features = torch.as_tensor(x, dtype=torch.float32)
        self.labels = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.features[index], self.labels[index]



#### CustomerRNN

In [32]:
%%inline_module src.pytorch_customer A06/src/pytorch_customer.py torch
class CustomerRNN(nn.Module):
    """One-layer Vanilla RNN followed by a single binary-classification logit."""

    def __init__(self, input_size: int = 5, hidden_size: int = 64, num_layers: int = 1) -> None:
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity="tanh",
        )
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        _, hidden = self.rnn(inputs)
        final_hidden = hidden[-1]
        return self.classifier(final_hidden).squeeze(-1)



#### count_trainable_parameters

In [33]:
%%inline_module src.pytorch_customer A06/src/pytorch_customer.py torch
def count_trainable_parameters(model: nn.Module) -> int:
    """Count parameters that participate in gradient updates."""

    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)



#### compute_positive_weight

In [34]:
%%inline_module src.pytorch_customer A06/src/pytorch_customer.py torch
def compute_positive_weight(train_labels: np.ndarray) -> float:
    """Return negative/positive ratio from the supplied TRAIN labels only."""

    labels = np.asarray(train_labels)
    negatives = int(np.count_nonzero(labels == 0))
    positives = int(np.count_nonzero(labels == 1))
    if negatives == 0 or positives == 0 or negatives + positives != len(labels):
        raise ValueError("train_labels must contain both binary classes and no other values")
    return negatives / positives



#### _average_loss

In [35]:
%%inline_module src.pytorch_customer A06/src/pytorch_customer.py torch
def _average_loss(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    model.eval()
    total_loss = 0.0
    total_samples = 0
    with torch.inference_mode():
        for features, labels in loader:
            features = features.to(device)
            labels = labels.to(device)
            logits = model(features)
            loss = criterion(logits, labels)
            batch_size = len(labels)
            total_loss += float(loss.item()) * batch_size
            total_samples += batch_size
    if total_samples == 0:
        raise ValueError("loader must contain at least one sample")
    return total_loss / total_samples



#### _save_checkpoint_atomic

In [36]:
%%inline_module src.pytorch_customer A06/src/pytorch_customer.py torch
def _save_checkpoint_atomic(path: Path, payload: dict[str, Any]) -> None:
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(f".{destination.name}.tmp")
    torch.save(payload, temporary)
    os.replace(temporary, destination)



#### train_with_early_stopping

In [37]:
%%inline_module src.pytorch_customer A06/src/pytorch_customer.py torch
def train_with_early_stopping(
    *,
    model: nn.Module,
    train_loader: DataLoader,
    validation_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    checkpoint_path: Path,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 1e-4,
    gradient_clip_norm: float | None = 1.0,
    seed: int = 42,
    verbose: bool = True,
) -> dict[str, Any]:
    """Train on TRAIN, select only by validation loss, and restore the best state."""

    if max_epochs <= 0 or patience <= 0:
        raise ValueError("max_epochs and patience must be positive")

    model.to(device)
    history: dict[str, list[float]] = {"train_loss": [], "validation_loss": []}
    best_validation_loss = float("inf")
    patience_reference_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    start_time = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss = 0.0
        sample_count = 0
        for features, labels in train_loader:
            features = features.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = criterion(logits, labels)
            loss.backward()
            if gradient_clip_norm is not None:
                nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)
            optimizer.step()
            batch_size = len(labels)
            running_loss += float(loss.item()) * batch_size
            sample_count += batch_size

        if sample_count == 0:
            raise ValueError("train_loader must contain at least one sample")
        train_loss = running_loss / sample_count
        validation_loss = _average_loss(model, validation_loader, criterion, device)
        history["train_loss"].append(train_loss)
        history["validation_loss"].append(validation_loss)

        new_best = validation_loss < best_validation_loss
        if new_best:
            best_validation_loss = validation_loss
            best_epoch = epoch
            _save_checkpoint_atomic(
                checkpoint_path,
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_epoch": best_epoch,
                    "best_validation_loss": best_validation_loss,
                    "seed": seed,
                    "architecture": {
                        "input_size": getattr(model, "input_size", None),
                        "hidden_size": getattr(model, "hidden_size", None),
                        "num_layers": getattr(model, "num_layers", None),
                    },
                },
            )

        meaningful_improvement = validation_loss < patience_reference_loss - min_delta
        if meaningful_improvement:
            patience_reference_loss = validation_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if verbose:
            marker = " *" if new_best else ""
            print(
                f"Epoch {epoch:02d}/{max_epochs} | train={train_loss:.6f} | "
                f"val={validation_loss:.6f}{marker}"
            )
        if epochs_without_improvement >= patience:
            if verbose:
                print(f"Early stopping tại epoch {epoch}; best epoch = {best_epoch}.")
            break

    duration_seconds = time.perf_counter() - start_time
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    return {
        "history": history,
        "epochs_ran": len(history["train_loss"]),
        "best_epoch": int(best_epoch),
        "best_validation_loss": float(best_validation_loss),
        "training_duration_seconds": float(duration_seconds),
    }



#### predict_probabilities

In [38]:
%%inline_module src.pytorch_customer A06/src/pytorch_customer.py torch
def predict_probabilities(
    model: nn.Module,
    loader: DataLoader,
    *,
    device: torch.device,
) -> tuple[np.ndarray, np.ndarray]:
    """Return true labels and sigmoid probabilities in DataLoader order."""

    model.eval()
    label_batches: list[np.ndarray] = []
    probability_batches: list[np.ndarray] = []
    with torch.inference_mode():
        for features, labels in loader:
            logits = model(features.to(device))
            probabilities = torch.sigmoid(logits)
            label_batches.append(labels.cpu().numpy())
            probability_batches.append(probabilities.cpu().numpy())
    if not label_batches:
        raise ValueError("loader must contain at least one sample")
    return np.concatenate(label_batches), np.concatenate(probability_batches)



#### binary_classification_metrics

In [39]:
%%inline_module src.pytorch_customer A06/src/pytorch_customer.py torch
def binary_classification_metrics(
    labels: np.ndarray,
    probabilities: np.ndarray,
    *,
    threshold: float = 0.5,
) -> dict[str, float]:
    """Calculate fixed-threshold classification metrics plus threshold-free ROC-AUC."""

    y_true = np.asarray(labels).astype(np.int8)
    y_probability = np.asarray(probabilities, dtype=np.float64)
    if y_true.shape != y_probability.shape:
        raise ValueError("labels and probabilities must have identical shapes")
    predictions = (y_probability >= threshold).astype(np.int8)
    return {
        "accuracy": float(accuracy_score(y_true, predictions)),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_probability)),
    }



## Thực hành: 02_Customer_Behavior_Data

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

**Bản đường dẫn r2.** Notebook gốc và output được giữ trong provenance. Kernel: Python3.11 WSL NumPy/PyTorch. Không Run All pipeline huấn luyện chỉ để xem kết quả đã lưu.


# 02 — Hiểu dữ liệu hành vi khách hàng theo thời gian

Notebook này thực hiện **data understanding** và **EDA** cho UCI Online Retail II. Mục tiêu là hiểu dữ liệu giao dịch, xác định cleaning phù hợp với bài toán purchase-behavior forecasting, và minh họa cách chuyển từ transaction lines sang customer-week observations.

Giới hạn của notebook:

- chỉ đọc workbook raw đã khóa tại `datasets/customer/online_retail_II.xlsx`;
- không sửa hoặc ghi đè workbook raw;
- không tạo train/validation/test sequences cuối cùng;
- không fit scaler, không tính class weight và không train PyTorch/Keras RNN.


## Thiết lập

`SEED = 42` được giữ nhất quán với experiment contract. Toàn bộ figures tái sử dụng được lưu vào `results/figures/customer_eda/`.


In [40]:
if RUN_EXPERIMENTS:
    import gc
    import random
    import sys
    from pathlib import Path

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from IPython.display import display

    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)

    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

    RAW_PATH = PROJECT_ROOT / "datasets" / "customer" / "online_retail_II.xlsx"
    FIGURE_DIR = PROJECT_ROOT / "results" / "figures" / "customer_eda"
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

    plt.style.use("seaborn-v0_8-whitegrid")
    pd.set_option("display.max_columns", 20)
    pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

    assert RAW_PATH.is_file(), f"Không tìm thấy raw workbook: {RAW_PATH}"
    print(f"Python executable: {sys.executable}")
    print(f"Raw workbook: {RAW_PATH}")
    print(f"Figure directory: {FIGURE_DIR}")

## 1. Giới thiệu dataset

**Online Retail II** ghi lại các transaction lines của một nhà bán lẻ trực tuyến tại Anh trong khoảng **01/12/2009–09/12/2011**. Mỗi dòng không nhất thiết là một đơn hàng: một invoice có thể chứa nhiều sản phẩm và vì vậy xuất hiện trên nhiều dòng.

Các trường chính mô tả invoice, sản phẩm, số lượng, thời điểm, đơn giá, khách hàng và quốc gia. Dữ liệu phù hợp với phân tích hành vi theo thời gian vì mỗi giao dịch có `InvoiceDate` và có thể liên kết các lần mua của cùng `CustomerID`.

Bài toán RNN ở milestone sau sẽ là:

> Từ 8 tuần hành vi gần nhất của một khách hàng, dự đoán khách hàng có mua trong tuần kế tiếp hay không.

Đây là binary classification ở mức **customer-week**, không phải dự đoán từng transaction line.


## 2. Đọc và xác thực các raw sheets

Ta hỏi workbook về sheet names trước, đọc từng sheet riêng, ghi lại shape/columns/dtypes, rồi chỉ `concat` sau khi xác nhận schema giống nhau. Không giả định tên cột theo tài liệu phiên bản mới.


In [41]:
if RUN_EXPERIMENTS:
    excel_file = pd.ExcelFile(RAW_PATH)
    sheet_names = excel_file.sheet_names
    print("Sheet names:", sheet_names)

    sheet_frames = {}
    sheet_records = []
    sheet_dtypes = {}

    for sheet_name in sheet_names:
        frame = pd.read_excel(RAW_PATH, sheet_name=sheet_name)
        sheet_frames[sheet_name] = frame
        sheet_records.append(
            {
                "sheet": sheet_name,
                "rows": len(frame),
                "columns": frame.shape[1],
                "column_names": ", ".join(map(str, frame.columns)),
            }
        )
        sheet_dtypes[sheet_name] = frame.dtypes.astype(str)

    sheet_summary = pd.DataFrame(sheet_records)
    dtype_summary = pd.DataFrame(sheet_dtypes).T
    display(sheet_summary)
    display(dtype_summary)

    schemas = [tuple(frame.columns) for frame in sheet_frames.values()]
    assert all(
        schema == schemas[0] for schema in schemas
    ), "Schemas giữa các sheets không tương thích"

    raw_df = pd.concat(sheet_frames.values(), ignore_index=True)
    print(f"Schema tương thích: {list(raw_df.columns)}")
    print(f"Combined raw shape: {raw_df.shape}")
    display(raw_df.head(3))

    del sheet_frames
    _ = gc.collect()

Hai sheets có cùng 8 raw columns và lần lượt 525.461 và 541.910 dòng, nên có thể ghép theo hàng. Tổng cộng là **1.067.371 transaction lines**.

Workbook dùng tên cột lịch sử. Ta chuẩn hóa **chỉ trong DataFrame memory**:

- `Invoice → InvoiceNo`
- `Price → UnitPrice`
- `Customer ID → CustomerID`

File `.xlsx` trên đĩa không bị thay đổi.


In [42]:
if RUN_EXPERIMENTS:
    expected_aliases = {
        "Invoice": "InvoiceNo",
        "Price": "UnitPrice",
        "Customer ID": "CustomerID",
    }
    rename_map = {
        raw_name: normalized_name
        for raw_name, normalized_name in expected_aliases.items()
        if raw_name in raw_df.columns
    }

    data_df = raw_df.rename(columns=rename_map).copy()
    required_columns = {
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country",
    }
    assert required_columns.issubset(data_df.columns)

    print("Column normalization thực tế:", rename_map)
    print("Columns sau normalization:", list(data_df.columns))

## 3. Data integrity audit

Audit bên dưới đo shape, exact duplicates, missing values, ngày lỗi, phân phối `Quantity`/`UnitPrice`, quy mô khách hàng/invoice/quốc gia và date range. Ta chưa lọc dữ liệu trong bước này để các vấn đề không bị che mất.


In [43]:
if RUN_EXPERIMENTS:
    data_df["InvoiceDate"] = pd.to_datetime(data_df["InvoiceDate"], errors="coerce")
    data_df["CustomerID"] = pd.to_numeric(data_df["CustomerID"], errors="coerce")
    data_df["Quantity"] = pd.to_numeric(data_df["Quantity"], errors="coerce")
    data_df["UnitPrice"] = pd.to_numeric(data_df["UnitPrice"], errors="coerce")

    invoice_text = data_df["InvoiceNo"].astype("string").str.strip()
    is_cancellation = invoice_text.str.upper().str.startswith("C", na=False)

    audit_overview = pd.Series(
        {
            "rows": len(data_df),
            "columns": data_df.shape[1],
            "exact_duplicate_rows": int(data_df.duplicated().sum()),
            "invalid_dates": int(data_df["InvoiceDate"].isna().sum()),
            "missing_customer_id": int(data_df["CustomerID"].isna().sum()),
            "unique_countries": int(data_df["Country"].nunique(dropna=True)),
            "unique_customers": int(data_df["CustomerID"].nunique(dropna=True)),
            "unique_invoices": int(data_df["InvoiceNo"].nunique(dropna=True)),
            "date_min": data_df["InvoiceDate"].min(),
            "date_max": data_df["InvoiceDate"].max(),
        },
        name="value",
    )
    missing_summary = (
        data_df.isna()
        .sum()
        .rename("missing_count")
        .to_frame()
        .assign(missing_pct=lambda table: 100 * table["missing_count"] / len(data_df))
    )

    display(audit_overview.to_frame())
    display(missing_summary)

In [44]:
if RUN_EXPERIMENTS:
    selected_percentiles = [0.01, 0.25, 0.50, 0.75, 0.99]
    distribution_summary = data_df[["Quantity", "UnitPrice"]].describe(
        percentiles=selected_percentiles
    )
    display(distribution_summary)

    print("Quantity < 0:", int((data_df["Quantity"] < 0).sum()))
    print("Quantity = 0:", int((data_df["Quantity"] == 0).sum()))
    print("UnitPrice <= 0:", int((data_df["UnitPrice"] <= 0).sum()))

**Nhận xét integrity thực tế**

- Có **34.335 exact duplicate rows**. Vì `Quantity` đã biểu diễn số lượng, giữ exact duplicates sẽ có nguy cơ đếm lặp doanh thu và hành vi; modeling DataFrame sẽ loại chúng nhưng audit raw vẫn giữ nguyên.
- `CustomerID` thiếu ở **243.007 dòng**; các dòng này không thể gắn thành sequence của một khách hàng cụ thể.
- `Description` thiếu ở **4.382 dòng**, nhưng không phải điều kiện quyết định vì feature production dùng `StockCode`, số lượng, invoice và revenue.
- Không có `InvoiceDate` lỗi/thiếu sau parsing.
- `Quantity` và `UnitPrice` có extreme values và giá trị âm/không dương. Vì vậy mean đơn thuần dễ bị chi phối; notebook hiển thị percentiles và dùng cleaning theo ý nghĩa mua hàng.
- Date range raw là **2009-12-01 07:45:00 đến 2011-12-09 12:50:00**; tháng đầu và tháng cuối là các khoảng quan sát không đầy đủ.


## 4. Cancellation invoices

Ta xác định convention từ chính dữ liệu: invoice bị hủy có mã bắt đầu bằng chữ `C` (không phân biệt hoa/thường). Cancellation thường đi kèm `Quantity < 0`, biểu thị đảo ngược giao dịch trước đó. Chúng mang thông tin nghiệp vụ về return/cancellation, nhưng không phải một lần **purchase dương** cho target “có mua tuần sau”.

Vì vậy ta đếm và quan sát chúng **trước khi lọc**, thay vì xóa im lặng.


In [45]:
if RUN_EXPERIMENTS:
    cancellation_summary = pd.Series(
        {
            "cancellation_lines": int(is_cancellation.sum()),
            "unique_cancellation_invoices": int(
                data_df.loc[is_cancellation, "InvoiceNo"].nunique(dropna=True)
            ),
            "cancellation_lines_with_negative_quantity": int(
                (is_cancellation & (data_df["Quantity"] < 0)).sum()
            ),
            "all_negative_quantity_lines": int((data_df["Quantity"] < 0).sum()),
        },
        name="count",
    )
    display(cancellation_summary.to_frame())
    display(
        data_df.loc[
            is_cancellation,
            [
                "InvoiceNo",
                "StockCode",
                "Description",
                "Quantity",
                "UnitPrice",
                "InvoiceDate",
            ],
        ].head(5)
    )

Kết quả có **19.494 cancellation lines** thuộc **8.292 cancellation invoices**; 19.493 dòng trong số đó có `Quantity < 0`. Việc kiểm tra cả prefix và dấu của `Quantity` giúp phát hiện một ngoại lệ thay vì giả định hai quy tắc hoàn toàn tương đương.


## 5. Cleaning rules cho purchase-behavior forecasting

Modeling DataFrame áp dụng các quy tắc sau trong memory:

1. bỏ exact duplicate rows để tránh đếm lặp cùng transaction line;
2. giữ `CustomerID` numeric, dương và nguyên để có thể tạo lịch sử theo khách hàng;
3. giữ `InvoiceDate` hợp lệ;
4. loại invoice có prefix `C` vì đó là cancellation, không phải purchase dương;
5. yêu cầu `Quantity > 0`;
6. yêu cầu `UnitPrice > 0`.

Các quy tắc này phục vụ task dự đoán **purchase activity**. Một bài toán riêng về return/cancellation có thể cần giữ các dòng âm, nhưng đó không phải target của A06.


In [46]:
if RUN_EXPERIMENTS:
    customer_is_integer = data_df["CustomerID"].mod(1).eq(0).fillna(False)
    valid_customer = (
        data_df["CustomerID"].notna() & data_df["CustomerID"].gt(0) & customer_is_integer
    )
    valid_date = data_df["InvoiceDate"].notna()
    positive_quantity = data_df["Quantity"].gt(0)
    positive_price = data_df["UnitPrice"].gt(0)

    deduplicated_df = data_df.drop_duplicates().copy()
    dedup_invoice_text = deduplicated_df["InvoiceNo"].astype("string").str.strip()
    dedup_is_cancellation = dedup_invoice_text.str.upper().str.startswith("C", na=False)
    dedup_customer_integer = deduplicated_df["CustomerID"].mod(1).eq(0).fillna(False)

    clean_mask = (
        deduplicated_df["CustomerID"].notna()
        & deduplicated_df["CustomerID"].gt(0)
        & dedup_customer_integer
        & deduplicated_df["InvoiceDate"].notna()
        & ~dedup_is_cancellation
        & deduplicated_df["Quantity"].gt(0)
        & deduplicated_df["UnitPrice"].gt(0)
    )

    cleaning_checks = pd.DataFrame(
        {
            "rule": [
                "raw rows",
                "exact duplicates removed",
                "invalid/missing CustomerID",
                "invalid InvoiceDate",
                "cancellation prefix C",
                "Quantity <= 0",
                "UnitPrice <= 0",
                "final cleaned rows",
            ],
            "count": [
                len(data_df),
                int(data_df.duplicated().sum()),
                int((~valid_customer).sum()),
                int((~valid_date).sum()),
                int(is_cancellation.sum()),
                int((~positive_quantity).sum()),
                int((~positive_price).sum()),
                int(clean_mask.sum()),
            ],
        }
    )

    clean_df = deduplicated_df.loc[clean_mask].copy()
    clean_df["InvoiceNo"] = clean_df["InvoiceNo"].astype("string").str.strip()
    clean_df["CustomerID"] = clean_df["CustomerID"].astype("int64")

    display(cleaning_checks)
    print(f"Cleaned modeling shape: {clean_df.shape}")
    print(f"Cleaned customers: {clean_df['CustomerID'].nunique():,}")
    print(f"Cleaned invoices: {clean_df['InvoiceNo'].nunique():,}")
    print(
        "Cleaned date range:",
        clean_df["InvoiceDate"].min(),
        "->",
        clean_df["InvoiceDate"].max(),
    )

    del raw_df, data_df, deduplicated_df
    _ = gc.collect()

Các failure counts trong bảng có thể chồng lấp nhau, nên không được cộng trực tiếp để suy ra số dòng cuối. `final cleaned rows` được tính bằng một combined mask duy nhất sau khi bỏ exact duplicates.

## 6. Tạo `Revenue`

Với các purchase lines hợp lệ:

$$
\text{Revenue} = \text{Quantity} \times \text{UnitPrice}
$$

Đây là giá trị ở mức invoice line. Khi tổng hợp customer-week, `total_spent` là tổng `Revenue` của các dòng trong tuần đó.


In [47]:
if RUN_EXPERIMENTS:
    clean_df["Revenue"] = clean_df["Quantity"] * clean_df["UnitPrice"]
    assert clean_df["Revenue"].gt(0).all()

    display(
        clean_df[
            [
                "InvoiceNo",
                "StockCode",
                "Quantity",
                "UnitPrice",
                "Revenue",
                "InvoiceDate",
                "CustomerID",
            ]
        ].head(5)
    )
    print(f"Tổng Revenue trong cleaned EDA data: {clean_df['Revenue'].sum():,.2f}")

## 7. Phân tích hành vi khách hàng

### 7.1 Orders, revenue và active customers theo tháng

“Order” được đếm bằng số `InvoiceNo` duy nhất, không phải số invoice lines. Ba đường thời gian dùng chung cleaned DataFrame nhưng đo ba khía cạnh khác nhau: lượng orders, giá trị mua hàng và độ rộng của customer base hoạt động.


In [48]:
if RUN_EXPERIMENTS:
    clean_df["Month"] = clean_df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
    monthly_activity = (
        clean_df.groupby("Month", observed=True)
        .agg(
            transaction_lines=("InvoiceNo", "size"),
            order_count=("InvoiceNo", "nunique"),
            revenue=("Revenue", "sum"),
            active_customers=("CustomerID", "nunique"),
        )
        .sort_index()
    )

    display(monthly_activity.head(4))
    display(monthly_activity.tail(4))

    fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
    axes[0].plot(
        monthly_activity.index, monthly_activity["order_count"], color="#2A6F97", marker="o"
    )
    axes[0].set_ylabel("Unique orders")
    axes[0].set_title("Hoạt động mua hàng theo tháng")

    axes[1].plot(
        monthly_activity.index, monthly_activity["revenue"], color="#D97706", marker="o"
    )
    axes[1].set_ylabel("Revenue")

    axes[2].plot(
        monthly_activity.index,
        monthly_activity["active_customers"],
        color="#2D6A4F",
        marker="o",
    )
    axes[2].set_ylabel("Active customers")
    axes[2].set_xlabel("Tháng")

    for axis in axes:
        axis.grid(alpha=0.25)

    fig.tight_layout()
    output_path = FIGURE_DIR / "monthly_activity.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Đã lưu: {output_path}")

**Nhận xét:** ba series thay đổi cùng chiều ở nhiều giai đoạn nhưng không hoàn toàn trùng nhau: nhiều orders không nhất thiết đồng nghĩa revenue tăng tỷ lệ thuận. Hoạt động mạnh vào các giai đoạn cuối năm cho thấy seasonality có thể quan trọng. Tuy nhiên tháng 12/2009 và 12/2011 là partial months, vì vậy không nên so sánh trực tiếp chúng với tháng đầy đủ.

### 7.2 Phân phối spending, purchase frequency và order count

Ta tóm tắt mỗi khách hàng bằng tổng spending, số orders duy nhất và số active weeks. `active_weeks` là một cách nhìn purchase frequency phù hợp với unit thời gian của task.


In [49]:
if RUN_EXPERIMENTS:
    clean_df["WeekStart"] = clean_df["InvoiceDate"].dt.to_period("W-SUN").dt.start_time
    customer_summary = clean_df.groupby("CustomerID", observed=True).agg(
        total_spent=("Revenue", "sum"),
        order_count=("InvoiceNo", "nunique"),
        active_weeks=("WeekStart", "nunique"),
    )

    display(customer_summary.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.99]))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    axes[0].hist(
        np.log10(1 + customer_summary["total_spent"]),
        bins=40,
        color="#D97706",
        edgecolor="white",
    )
    axes[0].set(
        title="Spending distribution", xlabel="log10(1 + total_spent)", ylabel="Customers"
    )

    axes[1].hist(
        customer_summary["active_weeks"], bins=35, color="#2D6A4F", edgecolor="white"
    )
    axes[1].set(title="Purchase frequency", xlabel="Active weeks", ylabel="Customers")

    axes[2].hist(
        np.log10(1 + customer_summary["order_count"]),
        bins=35,
        color="#2A6F97",
        edgecolor="white",
    )
    axes[2].set(
        title="Order count distribution",
        xlabel="log10(1 + unique orders)",
        ylabel="Customers",
    )

    for axis in axes:
        axis.grid(alpha=0.2)

    fig.tight_layout()
    output_path = FIGURE_DIR / "customer_distributions.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Đã lưu: {output_path}")

**Nhận xét:** các phân phối lệch phải rõ rệt—đa số khách hàng mua ít lần, trong khi một nhóm nhỏ mua thường xuyên và chi tiêu rất lớn. Vì vậy biểu đồ spending và order count dùng `log10(1 + value)` để phần thân phân phối vẫn đọc được; đây chỉ là cách vẽ EDA, không phải feature transformation đã khóa cho model.

### 7.3 Quốc gia

Ta so sánh top 10 quốc gia theo revenue, đồng thời xem số active customers. Hai chỉ số giúp tránh kết luận chỉ từ tổng tiền.


In [50]:
if RUN_EXPERIMENTS:
    country_summary = (
        clean_df.groupby("Country", observed=True)
        .agg(
            revenue=("Revenue", "sum"),
            order_count=("InvoiceNo", "nunique"),
            active_customers=("CustomerID", "nunique"),
        )
        .sort_values("revenue", ascending=False)
    )
    top_countries = country_summary.head(10).sort_values("revenue")
    display(country_summary.head(10))

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].barh(top_countries.index, top_countries["revenue"], color="#D97706")
    axes[0].set(title="Top 10 quốc gia theo Revenue", xlabel="Revenue")

    axes[1].barh(top_countries.index, top_countries["active_customers"], color="#2A6F97")
    axes[1].set(title="Active customers trong cùng top 10", xlabel="Customers")

    fig.tight_layout()
    output_path = FIGURE_DIR / "top_countries.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Đã lưu: {output_path}")

**Nhận xét:** `United Kingdom` chiếm ưu thế rất lớn về revenue và số khách hàng, phù hợp với bối cảnh nhà bán lẻ tại Anh. Các quốc gia khác vẫn có customer behavior đáng kể nhưng quy mô không đồng đều; model sau này không dùng `Country` trong bộ 5 features đã khóa.


## 8. Khái niệm customer-week aggregation

Một row trong bảng tuần biểu diễn hành vi của **một khách hàng trong một tuần hoạt động**. Quy ước tuần là thứ Hai–Chủ Nhật (`W-SUN`, nên `WeekStart` là thứ Hai).

Candidate features:

- `total_spent`: tổng `Revenue`;
- `total_quantity`: tổng số lượng mua;
- `order_count`: số `InvoiceNo` duy nhất;
- `unique_products`: số `StockCode` duy nhất;
- `active_flag = 1` nếu có ít nhất một purchase hợp lệ.

Điểm quan trọng: nhiều invoice lines cùng `InvoiceNo` vẫn chỉ là **một order**.


In [51]:
if RUN_EXPERIMENTS:
    customer_week_active = (
        clean_df.groupby(["CustomerID", "WeekStart"], observed=True)
        .agg(
            total_spent=("Revenue", "sum"),
            total_quantity=("Quantity", "sum"),
            order_count=("InvoiceNo", "nunique"),
            unique_products=("StockCode", "nunique"),
        )
        .reset_index()
    )
    customer_week_active["active_flag"] = np.int8(1)

    assert customer_week_active["order_count"].ge(1).all()
    assert customer_week_active["active_flag"].eq(1).all()

    print(f"Active customer-week rows: {len(customer_week_active):,}")
    print(
        "WeekStart range:",
        customer_week_active["WeekStart"].min().date(),
        "->",
        customer_week_active["WeekStart"].max().date(),
    )
    display(customer_week_active.head(8))

Bảng trên chỉ chứa **active weeks**. Nó là bước trung gian để giải thích aggregation, chưa phải production sequence dataset và chưa được lưu thành train/validation/test files.

## 9. Missing/inactive weeks

RNN cần các time steps cách đều nhau. Nếu một khách hàng mua ở tuần 1 và tuần 3, bỏ tuần 2 sẽ khiến model tưởng hai lần mua liên tiếp nhau. Vì vậy cần reindex lịch tuần đều và điền tuần không mua bằng zero cho cả 5 behavior features.

Demo sau chọn một khách hàng có đủ active weeks và có gap, rồi chỉ hiển thị một cửa sổ nhỏ quanh inactive week đầu tiên.


In [52]:
if RUN_EXPERIMENTS:
    customer_span = customer_week_active.groupby("CustomerID", observed=True)[
        "WeekStart"
    ].agg(first_week="min", last_week="max", active_weeks="count")
    customer_span["span_weeks"] = (
        customer_span["last_week"] - customer_span["first_week"]
    ).dt.days // 7 + 1
    customer_span["inactive_weeks"] = (
        customer_span["span_weeks"] - customer_span["active_weeks"]
    )

    demo_candidates = customer_span.query("active_weeks >= 8 and inactive_weeks >= 3")
    demo_customer = int(
        demo_candidates.sort_values(
            ["inactive_weeks", "active_weeks"], ascending=[False, False]
        ).index[0]
    )

    demo_active = customer_week_active.loc[
        customer_week_active["CustomerID"].eq(demo_customer)
    ].set_index("WeekStart")
    demo_index = pd.date_range(
        demo_active.index.min(), demo_active.index.max(), freq="W-MON"
    )
    demo_timeline = demo_active.reindex(demo_index)
    feature_columns = [
        "total_spent",
        "total_quantity",
        "order_count",
        "unique_products",
        "active_flag",
    ]
    demo_timeline[feature_columns] = demo_timeline[feature_columns].fillna(0)
    demo_timeline["CustomerID"] = demo_customer
    demo_timeline.index.name = "WeekStart"

    first_gap_position = np.flatnonzero(demo_timeline["active_flag"].eq(0).to_numpy())[0]
    window_start = max(0, first_gap_position - 3)
    demo_window = demo_timeline.iloc[window_start : first_gap_position + 6].copy()

    print(f"Demo CustomerID: {demo_customer}")
    display(demo_window[["CustomerID", *feature_columns]])

    fig, ax = plt.subplots(figsize=(10, 4.5))
    colors = np.where(demo_window["active_flag"].eq(1), "#2A6F97", "#CBD5E1")
    ax.bar(demo_window.index, demo_window["total_spent"], width=5, color=colors)
    inactive_window = demo_window["active_flag"].eq(0)
    ax.scatter(
        demo_window.index[inactive_window],
        np.zeros(inactive_window.sum()),
        marker="x",
        s=55,
        color="#64748B",
        label="inactive week = 0",
        clip_on=False,
        zorder=3,
    )
    ax.set(
        title=f"Tuần hoạt động và không hoạt động — CustomerID {demo_customer}",
        xlabel="WeekStart",
        ylabel="total_spent",
    )
    ax.grid(axis="y", alpha=0.25)
    ax.set_ylim(bottom=-5)
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    output_path = FIGURE_DIR / "inactive_week_example.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Đã lưu: {output_path}")

Các cột hành vi của inactive week đều bằng 0. `CustomerID` vẫn được giữ để group/index sequence, nhưng sẽ **không** trở thành numerical input feature.

## 10. Target dự đoán tuần kế tiếp

Với target week $T$, input là 8 tuần ngay trước đó:

```text
[T-8, T-7, ..., T-1]  ──>  target tại T
```

- `target = 1`: khách hàng có ít nhất một purchase hợp lệ trong tuần $T$;
- `target = 0`: khách hàng không mua trong tuần $T$.

Target chỉ có hai trạng thái nên đây là **binary classification**. Không được đưa behavior của tuần $T$ vào 8 tuần features dùng để dự đoán chính tuần $T$.


## 11. Class distribution preview

Preview dưới đây chỉ ước lượng imbalance, chưa tạo tensor sequences. Quy ước prototype:

- loại hai boundary weeks không đầy đủ của workbook;
- timeline của mỗi khách hàng bắt đầu không sớm hơn purchase week đầu tiên;
- một target week chỉ hợp lệ sau khi đã có 8 tuần lịch sử;
- kéo timeline đến full week cuối để inactive periods sau lần mua cuối vẫn tạo negative targets.

Preview dùng toàn bộ thời gian **chỉ cho EDA**. Không dùng tỷ lệ này để tạo class weight; class weight production phải tính riêng từ train labels sau chronological split.


In [53]:
if RUN_EXPERIMENTS:
    first_observed_week = customer_week_active["WeekStart"].min()
    last_observed_week = customer_week_active["WeekStart"].max()
    first_full_week = first_observed_week + pd.Timedelta(weeks=1)
    last_full_week = last_observed_week - pd.Timedelta(weeks=1)
    complete_weeks = pd.date_range(first_full_week, last_full_week, freq="W-MON")

    customer_ids = customer_week_active["CustomerID"].unique()
    prototype_grid = pd.MultiIndex.from_product(
        [customer_ids, complete_weeks], names=["CustomerID", "WeekStart"]
    ).to_frame(index=False)

    active_pairs = customer_week_active[["CustomerID", "WeekStart", "active_flag"]]
    prototype_grid = prototype_grid.merge(
        active_pairs, on=["CustomerID", "WeekStart"], how="left"
    )
    prototype_grid["active_flag"] = prototype_grid["active_flag"].fillna(0).astype("int8")

    customer_start = (
        customer_week_active.groupby("CustomerID", observed=True)["WeekStart"]
        .min()
        .clip(lower=first_full_week)
    )
    prototype_grid["timeline_start"] = prototype_grid["CustomerID"].map(customer_start)
    eligible_target = prototype_grid["WeekStart"] >= prototype_grid[
        "timeline_start"
    ] + pd.Timedelta(weeks=8)
    prototype_labels = prototype_grid.loc[eligible_target, "active_flag"]

    label_distribution = (
        prototype_labels.value_counts()
        .reindex([0, 1], fill_value=0)
        .rename_axis("target")
        .to_frame("count")
    )
    label_distribution["percentage"] = (
        100 * label_distribution["count"] / len(prototype_labels)
    )
    display(label_distribution)
    print(f"Prototype samples: {len(prototype_labels):,}")
    print(f"Positive rate: {prototype_labels.mean():.2%}")
    print(
        f"Complete target-week range: {first_full_week.date()} -> {last_full_week.date()}"
    )

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    bars = ax.bar(
        ["0 — không mua", "1 — có mua"],
        label_distribution["count"],
        color=["#94A3B8", "#2A6F97"],
    )
    ax.set(
        title="Prototype next-week label distribution", ylabel="Số customer-week targets"
    )
    for bar, percentage in zip(bars, label_distribution["percentage"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{percentage:.2f}%",
            ha="center",
            va="bottom",
        )
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    output_path = FIGURE_DIR / "prototype_label_balance.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Đã lưu: {output_path}")

    del prototype_grid, prototype_labels
    _ = gc.collect()

Preview cho thấy positive class chỉ khoảng **6,39%**, vì phần lớn customer-weeks không có purchase. Đây là dấu hiệu class imbalance đáng kể. Ở Prompt 5 cần khóa chính xác eligibility/window policy trước, split theo target week, rồi mới tính class weight bằng **train labels mà thôi**.


## Thực hành: 04_PyTorch_Customer_RNN

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

**Bản đường dẫn r2.** Notebook gốc và output được giữ trong provenance. Kernel: Python3.11 WSL NumPy/PyTorch. Không Run All pipeline huấn luyện chỉ để xem kết quả đã lưu.


# 04 — PyTorch Vanilla RNN dự đoán hành vi mua tuần kế tiếp

Notebook này huấn luyện một **Vanilla RNN** cho bài toán:

```text
8 customer-weeks trước đó × 5 behavior features
                    ↓
khách hàng có mua trong tuần kế tiếp hay không?
```

Phạm vi được khóa chặt:

- chỉ đọc ba artifact `customer_train.npz`, `customer_val.npz`, `customer_test.npz` đã hoàn tất ở preprocessing;
- không đọc lại workbook raw, không tạo lại sequence, không fit lại scaler và không thay đổi split;
- validation loss quyết định early stopping và checkpoint tốt nhất;
- TEST chỉ được đánh giá **một lần** sau khi đã khôi phục best validation state.


## 1. Thiết lập và reproducibility

`SEED = 42` được áp dụng cho Python, NumPy và PyTorch. Máy hiện tại không có CUDA nên thí nghiệm chạy trên CPU. `torch.use_deterministic_algorithms(True)` tăng tính lặp lại trên cùng môi trường; điều này không phải lời hứa bit-for-bit giữa phần cứng hoặc framework khác nhau.


In [54]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    import hashlib
    import json
    import os
    import sys
    from pathlib import Path

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import torch
    from IPython.display import display
    from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve
    from torch import nn
    from torch.utils.data import DataLoader

    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

    from src.config import SEED
    from src.preprocessing_utils import sha256_file
    from src.pytorch_customer import (
        CustomerRNN,
        CustomerSequenceDataset,
        binary_classification_metrics,
        compute_positive_weight,
        count_trainable_parameters,
        predict_probabilities,
        train_with_early_stopping,
    )
    from src.reproducibility import set_global_seed

    set_global_seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.use_deterministic_algorithms(True)
    torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    PROCESSED_DIR = PROJECT_ROOT / "datasets" / "processed"
    METADATA_PATH = PROJECT_ROOT / "results" / "metrics" / "preprocessing_metadata.json"
    FIGURE_DIR = PROJECT_ROOT / "results" / "figures" / "pytorch_customer"
    MODEL_DIR = PROJECT_ROOT / "models" / "pytorch"
    METRICS_PATH = PROJECT_ROOT / "results" / "metrics" / "pytorch_customer_metrics.json"
    PREDICTIONS_PATH = (
        PROJECT_ROOT / "results" / "predictions" / "pytorch_customer_predictions.csv"
    )
    CHECKPOINT_PATH = MODEL_DIR / "pytorch_customer_rnn_best.pt"

    for directory in (FIGURE_DIR, MODEL_DIR, METRICS_PATH.parent, PREDICTIONS_PATH.parent):
        directory.mkdir(parents=True, exist_ok=True)

    plt.style.use("seaborn-v0_8-whitegrid")
    print("Python executable:", sys.executable)
    print("PyTorch version:  ", torch.__version__)
    print("Device:           ", DEVICE)

## 2. Nạp đúng shared artifacts

Notebook đọc đường dẫn và SHA-256 từ `preprocessing_metadata.json`, rồi kiểm tra hash trước khi dùng. Mỗi artifact phải có:

- `X`: tensor đã được scale, shape `(samples, 8, 5)`;
- `y`: nhãn nhị phân của tuần kế tiếp;
- `customer_id`, `target_week`: metadata để truy vết prediction, **không** đưa vào model.

Việc kiểm tra này ngăn notebook âm thầm dùng một bản preprocessing khác với Keras sau này.


In [55]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    preprocessing_metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
    customer_metadata = preprocessing_metadata["customer"]
    feature_names = customer_metadata["feature_names"]

    assert feature_names == [
        "total_spent",
        "total_quantity",
        "order_count",
        "unique_products",
        "active_flag",
    ]
    assert "CustomerID" not in feature_names
    assert customer_metadata["sequence_length"] == 8

    splits = {}
    summary_rows = []
    for split_name in ("train", "val", "test"):
        artifact_info = customer_metadata["artifacts"][split_name]
        artifact_path = PROJECT_ROOT / artifact_info["path"]
        assert sha256_file(artifact_path) == artifact_info["sha256"]

        with np.load(artifact_path, allow_pickle=False) as artifact:
            split = {key: artifact[key].copy() for key in artifact.files}
        expected_shape = tuple(customer_metadata["splits"][split_name]["X_shape"])
        assert split["X"].shape == expected_shape
        assert split["X"].shape[1:] == (8, 5)
        assert split["y"].shape == (len(split["X"]),)
        assert len(split["customer_id"]) == len(split["target_week"]) == len(split["y"])
        assert set(np.unique(split["y"])) == {0, 1}
        splits[split_name] = split

        negatives = int(np.count_nonzero(split["y"] == 0))
        positives = int(np.count_nonzero(split["y"] == 1))
        summary_rows.append(
            {
                "split": split_name,
                "X shape": str(split["X"].shape),
                "class 0": negatives,
                "class 1": positives,
                "positive rate": positives / len(split["y"]),
                "target min": np.datetime_as_string(split["target_week"].min(), unit="D"),
                "target max": np.datetime_as_string(split["target_week"].max(), unit="D"),
            }
        )

    split_summary = pd.DataFrame(summary_rows).set_index("split")
    display(split_summary.style.format({"positive rate": "{:.2%}"}))

Class 1 chỉ chiếm khoảng 5–7% tùy split. Một classifier luôn đoán “không mua” vẫn có accuracy rất cao, vì vậy accuracy phải được đọc cùng precision, recall, F1 và ROC-AUC.

## 3. Dataset và DataLoader

`CustomerSequenceDataset` chỉ bọc các arrays đã xử lý thành tensor `float32`; nó không sửa feature hay label. `DataLoader` chia dữ liệu thành mini-batches:

- TRAIN dùng `shuffle=True` để optimizer không luôn thấy samples theo cùng thứ tự;
- validation và TEST dùng `shuffle=False` để evaluation ổn định và giữ đúng thứ tự metadata;
- shuffle chỉ đổi thứ tự đưa sample vào optimizer, **không** thay đổi chronological split đã khóa.


In [56]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    BATCH_SIZE = 1024

    datasets = {
        name: CustomerSequenceDataset(data["X"], data["y"]) for name, data in splits.items()
    }
    train_generator = torch.Generator().manual_seed(SEED)
    loaders = {
        "train": DataLoader(
            datasets["train"],
            batch_size=BATCH_SIZE,
            shuffle=True,
            generator=train_generator,
            num_workers=0,
        ),
        "val": DataLoader(
            datasets["val"], batch_size=BATCH_SIZE, shuffle=False, num_workers=0
        ),
        "test": DataLoader(
            datasets["test"], batch_size=BATCH_SIZE, shuffle=False, num_workers=0
        ),
    }

    batch_x, batch_y = next(iter(loaders["train"]))
    print("TRAIN batches:", len(loaders["train"]))
    print("Một batch X:", tuple(batch_x.shape), batch_x.dtype)
    print("Một batch y:", tuple(batch_y.shape), batch_y.dtype)

## 4. Vanilla RNN và forward pass

Model dùng một lớp `torch.nn.RNN` với `batch_first=True`:

```text
X: (batch, 8 weeks, 5 features)
            ↓ nn.RNN, hidden_size=64
h_n[-1]: (batch, 64)
            ↓ Linear(64, 1)
logit: (batch,)
```

Ở mỗi tuần, hidden state cập nhật từ feature tuần hiện tại và hidden state trước đó. Sau time step thứ 8, `h_n[-1]` là biểu diễn nén của toàn bộ 8 tuần theo thứ tự. Lớp `Linear` biến biểu diễn này thành **một logit**.

Logit chưa phải xác suất và có thể nhận mọi giá trị thực. Model cố ý **không chứa Sigmoid** vì `BCEWithLogitsLoss` kết hợp Sigmoid và binary cross-entropy theo cách ổn định số học hơn.


In [57]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    INPUT_SIZE = 5
    HIDDEN_SIZE = 64
    NUM_LAYERS = 1

    model = CustomerRNN(
        input_size=INPUT_SIZE,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
    ).to(DEVICE)
    parameter_count = count_trainable_parameters(model)

    with torch.inference_mode():
        sample_logits = model(batch_x[:4].to(DEVICE))

    print(model)
    print(f"Trainable parameters: {parameter_count:,}")
    print("Sample logits shape:", tuple(sample_logits.shape))
    assert parameter_count == 4609
    assert sample_logits.shape == (4,)

## 5. Loss, class imbalance và optimizer

Với logit $z$ và nhãn $y\in\{0,1\}$, `BCEWithLogitsLoss` đo độ sai của binary prediction. Vì positive class hiếm, ta dùng:

$$\text{pos\_weight}=\frac{N_{negative,\ TRAIN}}{N_{positive,\ TRAIN}}$$

Giá trị này chỉ được tính từ `train y`; validation và TEST không tham gia. Nó tăng chi phí khi bỏ sót positive sample, nhưng không tự động bảo đảm precision/recall tốt—kết quả vẫn phải được đo trên TEST.

Optimizer là Adam với learning rate nhỏ. Sau `backward()`, gradient norm được clip ở 1.0 để hạn chế exploding gradient của Vanilla RNN.


In [58]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    positive_weight = compute_positive_weight(splits["train"]["y"])
    metadata_positive_weight = customer_metadata["class_weight"]["positive_weight"]
    assert np.isclose(positive_weight, metadata_positive_weight)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([positive_weight], dtype=torch.float32, device=DEVICE)
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    print(f"TRAIN negatives / positives = {positive_weight:.6f}")
    print("Loss: BCEWithLogitsLoss với TRAIN-only pos_weight")
    print("Optimizer: Adam(lr=0.001)")

## 6. Training và early stopping

Mỗi epoch gồm hai pha:

1. TRAIN: forward → loss → backward → clip gradient → Adam update.
2. Validation: chỉ forward, không gradient và không update.

Checkpoint được thay khi validation loss đạt một raw minimum mới. `min_delta` chỉ quy định mức cải thiện đủ lớn để reset bộ đếm patience; vì vậy một cải thiện rất nhỏ vẫn được giữ làm best state nhưng không kéo dài training vô hạn. Nếu 5 epoch liên tiếp không có cải thiện đủ lớn, training dừng sớm. Sau cùng model tự khôi phục state của best epoch; TEST hoàn toàn không tham gia quyết định này.


In [59]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    MAX_EPOCHS = 30
    PATIENCE = 5
    MIN_DELTA = 1e-4
    GRADIENT_CLIP_NORM = 1.0

    training_result = train_with_early_stopping(
        model=model,
        train_loader=loaders["train"],
        validation_loader=loaders["val"],
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        checkpoint_path=CHECKPOINT_PATH,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        min_delta=MIN_DELTA,
        gradient_clip_norm=GRADIENT_CLIP_NORM,
        seed=SEED,
        verbose=True,
    )

    print(f"Epochs đã chạy: {training_result['epochs_ran']}")
    print(f"Best epoch: {training_result['best_epoch']}")
    print(f"Best validation loss: {training_result['best_validation_loss']:.6f}")
    print(f"Training duration: {training_result['training_duration_seconds']:.2f} giây")
    print("Best checkpoint:", CHECKPOINT_PATH)

## 7. Learning curves

Khoảng cách giữa train loss và validation loss giúp nhận biết underfitting/overfitting. Đường validation được dùng cho early stopping; dấu chấm đánh dấu epoch có validation loss thấp nhất.


In [60]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    history = training_result["history"]
    epochs = np.arange(1, training_result["epochs_ran"] + 1)

    fig, ax = plt.subplots(figsize=(8.5, 5))
    ax.plot(epochs, history["train_loss"], marker="o", label="TRAIN loss")
    ax.plot(epochs, history["validation_loss"], marker="o", label="Validation loss")
    best_epoch = training_result["best_epoch"]
    ax.scatter(
        best_epoch,
        history["validation_loss"][best_epoch - 1],
        color="#DC2626",
        s=70,
        zorder=3,
        label=f"Best epoch = {best_epoch}",
    )
    ax.set(
        title="PyTorch Customer RNN — learning curves",
        xlabel="Epoch",
        ylabel="Weighted BCE loss",
    )
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    loss_figure_path = FIGURE_DIR / "training_validation_loss.png"
    fig.savefig(loss_figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Đã lưu:", loss_figure_path)

## 8. Đánh giá TEST đúng một lần

Model hiện tại đã là best validation-state. Cell dưới đây thực hiện lần inference duy nhất trên TEST, chuyển logits thành xác suất bằng Sigmoid **ở ngoài model**, và dùng threshold cố định 0.5. Threshold không được tinh chỉnh bằng TEST.

Các metric:

- **Accuracy:** tỷ lệ dự đoán đúng, nhưng dễ gây hiểu nhầm khi class 0 chiếm đa số.
- **Precision:** trong các dự đoán “sẽ mua”, bao nhiêu trường hợp thực sự mua.
- **Recall:** trong các trường hợp thực sự mua, model tìm được bao nhiêu.
- **F1:** harmonic mean của precision và recall.
- **ROC-AUC:** khả năng xếp positive sample cao hơn negative sample trên mọi threshold.


In [61]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    test_labels, test_probabilities = predict_probabilities(
        model,
        loaders["test"],
        device=DEVICE,
    )
    THRESHOLD = 0.5
    test_predictions = (test_probabilities >= THRESHOLD).astype(np.int8)
    test_metrics = binary_classification_metrics(
        test_labels,
        test_probabilities,
        threshold=THRESHOLD,
    )
    test_confusion = confusion_matrix(test_labels, test_predictions, labels=[0, 1])

    metrics_table = pd.DataFrame(
        {"metric": list(test_metrics), "value": list(test_metrics.values())}
    ).set_index("metric")
    display(metrics_table.style.format("{:.4f}"))
    display(
        pd.DataFrame(
            test_confusion,
            index=["True 0", "True 1"],
            columns=["Pred 0", "Pred 1"],
        )
    )

    predictions = pd.DataFrame(
        {
            "CustomerID": splits["test"]["customer_id"].astype(np.int64),
            "target_week": np.datetime_as_string(splits["test"]["target_week"], unit="D"),
            "true_label": test_labels.astype(np.int8),
            "probability": test_probabilities.astype(np.float64),
            "predicted_label": test_predictions,
        }
    )
    assert len(predictions) == len(splits["test"]["y"])
    assert np.array_equal(predictions["true_label"].to_numpy(), splits["test"]["y"])
    predictions.to_csv(PREDICTIONS_PATH, index=False, float_format="%.8f")

    print(f"Đã lưu {len(predictions):,} TEST predictions: {PREDICTIONS_PATH}")
    display(predictions.head())

## 9. Confusion matrix

Confusion matrix cho biết trực tiếp số true negatives, false positives, false negatives và true positives. Với dữ liệu lệch lớp, đây là bổ sung quan trọng cho accuracy: model có thể tăng recall bằng cách chấp nhận nhiều false positives hơn.


In [62]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    display_cm = ConfusionMatrixDisplay(
        confusion_matrix=test_confusion,
        display_labels=["0 — không mua", "1 — có mua"],
    )
    display_cm.plot(ax=ax, cmap="Blues", colorbar=False, values_format=",d")
    ax.set_title("PyTorch Customer RNN — TEST confusion matrix")
    fig.tight_layout()
    confusion_figure_path = FIGURE_DIR / "test_confusion_matrix.png"
    fig.savefig(confusion_figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Đã lưu:", confusion_figure_path)

## 10. ROC curve

ROC curve quét qua nhiều thresholds và so sánh true-positive rate với false-positive rate. Đường chéo là ranking ngẫu nhiên; ROC-AUC càng gần 1 thì khả năng phân hạng càng tốt. ROC-AUC không thay thế precision/recall tại threshold 0.5, đặc biệt khi positive class hiếm.


In [63]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    false_positive_rate, true_positive_rate, _ = roc_curve(test_labels, test_probabilities)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(
        false_positive_rate,
        true_positive_rate,
        color="#2563EB",
        linewidth=2,
        label=f"RNN (AUC = {test_metrics['roc_auc']:.4f})",
    )
    ax.plot([0, 1], [0, 1], linestyle="--", color="#64748B", label="Random ranking")
    ax.set(
        title="PyTorch Customer RNN — TEST ROC curve",
        xlabel="False Positive Rate",
        ylabel="True Positive Rate",
        xlim=(0, 1),
        ylim=(0, 1.01),
    )
    ax.legend(loc="lower right")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    roc_figure_path = FIGURE_DIR / "test_roc_curve.png"
    fig.savefig(roc_figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Đã lưu:", roc_figure_path)

## 11. Lưu metrics và provenance

JSON kết quả ghi lại architecture, training protocol, class weighting, duration, TEST metrics và checksum của các input/output artifacts. Điều này cho phép Notebook 08 so sánh framework trên đúng cùng samples mà không phải suy đoán lại cấu hình.


In [64]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'torch':
    checkpoint_hash = sha256_file(CHECKPOINT_PATH)
    predictions_hash = sha256_file(PREDICTIONS_PATH)

    metrics_payload = {
        "experiment": "PyTorch Customer Vanilla RNN",
        "seed": SEED,
        "device": str(DEVICE),
        "architecture": {
            "recurrent_layer": "torch.nn.RNN",
            "input_size": INPUT_SIZE,
            "sequence_length": 8,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "batch_first": True,
            "nonlinearity": "tanh",
            "head": "Linear(64, 1) producing one logit",
            "sigmoid_inside_model": False,
            "parameter_count": parameter_count,
        },
        "data": {
            "feature_names": feature_names,
            "split_shapes": {name: list(data["X"].shape) for name, data in splits.items()},
            "source_artifacts": customer_metadata["artifacts"],
        },
        "training": {
            "optimizer": "Adam",
            "learning_rate": 1e-3,
            "criterion": "BCEWithLogitsLoss",
            "batch_size": BATCH_SIZE,
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "min_delta": MIN_DELTA,
            "gradient_clip_norm": GRADIENT_CLIP_NORM,
            "epochs_actually_run": training_result["epochs_ran"],
            "best_epoch": training_result["best_epoch"],
            "best_validation_loss": training_result["best_validation_loss"],
            "training_duration_seconds": training_result["training_duration_seconds"],
            "selection_scope": "validation loss only; TEST unused until final evaluation",
            "history": history,
        },
        "class_weighting": {
            "used": True,
            "type": "positive class weight in BCEWithLogitsLoss",
            "positive_weight": positive_weight,
            "formula": "TRAIN negatives / TRAIN positives",
            "fit_scope": "TRAIN y only",
            "train_class_0": int(np.count_nonzero(splits["train"]["y"] == 0)),
            "train_class_1": int(np.count_nonzero(splits["train"]["y"] == 1)),
        },
        "evaluation": {
            "split": "TEST",
            "evaluations_on_test": 1,
            "threshold": THRESHOLD,
            "metrics": test_metrics,
            "confusion_matrix_tn_fp_fn_tp": {
                "tn": int(test_confusion[0, 0]),
                "fp": int(test_confusion[0, 1]),
                "fn": int(test_confusion[1, 0]),
                "tp": int(test_confusion[1, 1]),
            },
        },
        "artifacts": {
            "checkpoint": {
                "path": CHECKPOINT_PATH.relative_to(PROJECT_ROOT).as_posix(),
                "sha256": checkpoint_hash,
            },
            "predictions": {
                "path": PREDICTIONS_PATH.relative_to(PROJECT_ROOT).as_posix(),
                "rows": len(predictions),
                "sha256": predictions_hash,
            },
            "figures": [
                loss_figure_path.relative_to(PROJECT_ROOT).as_posix(),
                confusion_figure_path.relative_to(PROJECT_ROOT).as_posix(),
                roc_figure_path.relative_to(PROJECT_ROOT).as_posix(),
            ],
        },
    }
    METRICS_PATH.write_text(
        json.dumps(metrics_payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

    print("Metrics JSON:", METRICS_PATH)
    print("Checkpoint SHA-256:", checkpoint_hash)
    print("Predictions SHA-256:", predictions_hash)

## Thực hành: 06_Keras_Customer_RNN

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

**Bản đường dẫn r2.** Notebook gốc và output được giữ trong provenance. Kernel: Python3.11 WSL TensorFlow/Keras. Không Run All pipeline huấn luyện chỉ để xem kết quả đã lưu.


# A06 — Keras Vanilla RNN dự báo hành vi mua hàng

Notebook này huấn luyện baseline Keras cho đúng bài toán đã khóa:

$$
\underbrace{8\ \text{tuần hành vi trước} \times 5\ \text{đặc trưng}}_{X}
\longrightarrow
\underbrace{\text{có mua trong tuần kế tiếp hay không}}_{y \in \{0,1\}}.
$$

Mục tiêu là triển khai `SimpleRNN` có ý nghĩa tương đương baseline PyTorch, không thay đổi dữ liệu và chưa kết luận framework nào tốt hơn. Notebook chỉ đọc ba artifact dùng chung; không tạo sequence, không chia lại split và không fit lại scaler.

## 1. Thiết lập tái lập

`SEED = 42` được áp dụng cho Python, NumPy và TensorFlow/Keras. Deterministic operations được yêu cầu khi runtime hỗ trợ; điều này giúp giảm dao động trong cùng môi trường nhưng không hứa hẹn kết quả bit-for-bit giữa phần cứng hoặc phiên bản thư viện khác nhau.

In [65]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    from pathlib import Path
    import hashlib
    import json
    import os
    import random
    import sys
    import time

    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
    os.environ["TF_DETERMINISTIC_OPS"] = "1"

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve
    import tensorflow as tf
    from tensorflow import keras

    ROOT = Path.cwd().resolve()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

    from src.keras_customer import (
        binary_classification_metrics,
        build_callbacks,
        build_customer_rnn,
        compute_class_weight,
        make_sample_weights,
        validate_customer_arrays,
    )

    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    keras.utils.set_random_seed(SEED)
    tf.config.experimental.enable_op_determinism()

    METADATA_PATH = ROOT / "results" / "metrics" / "preprocessing_metadata.json"
    PYTORCH_METRICS_PATH = ROOT / "results" / "metrics" / "pytorch_customer_metrics.json"
    MODEL_PATH = ROOT / "models" / "keras" / "keras_customer_rnn_best.keras"
    METRICS_PATH = ROOT / "results" / "metrics" / "keras_customer_metrics.json"
    PREDICTIONS_PATH = ROOT / "results" / "predictions" / "keras_customer_predictions.csv"
    FIGURE_DIR = ROOT / "results" / "figures" / "keras_customer"

    for directory in (MODEL_PATH.parent, METRICS_PATH.parent, PREDICTIONS_PATH.parent, FIGURE_DIR):
        directory.mkdir(parents=True, exist_ok=True)

    devices = tf.config.list_physical_devices()
    device_label = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
    print(f"Project root: {ROOT}")
    print(f"TensorFlow: {tf.__version__} | Keras: {keras.__version__} | device: {device_label} | seed: {SEED}")

## 2. Nạp đúng dữ liệu dùng chung và kiểm tra phép so sánh công bằng

Mỗi NPZ phải chứa đúng `X`, `y`, `customer_id`, `target_week`. Checksum được đối chiếu với metadata tiền xử lý, shape phải là `(samples, 8, 5)`, và ranh giới `target_week` phải theo thứ tự TRAIN → validation → TEST.

Năm feature theo thứ tự cố định là `total_spent`, `total_quantity`, `order_count`, `unique_products`, `active_flag`. `CustomerID` và `target_week` chỉ phục vụ truy vết, hoàn toàn không đi vào tensor đầu vào.

In [66]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    def sha256(path: Path) -> str:
        return hashlib.sha256(path.read_bytes()).hexdigest()


    metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
    pytorch_metrics = json.loads(PYTORCH_METRICS_PATH.read_text(encoding="utf-8"))
    customer_meta = metadata["customer"]

    assert metadata["seed"] == pytorch_metrics["seed"] == SEED
    assert customer_meta["sequence_length"] == 8
    assert customer_meta["feature_names"] == pytorch_metrics["data"]["feature_names"]
    assert pytorch_metrics["evaluation"]["threshold"] == 0.5

    required_keys = {"X", "y", "customer_id", "target_week"}
    artifacts = {}
    artifact_hashes = {}
    for split in ("train", "val", "test"):
        info = customer_meta["artifacts"][split]
        path = ROOT / info["path"]
        actual_hash = sha256(path)
        assert actual_hash == info["sha256"], f"Checksum sai: {path}"
        with np.load(path) as loaded:
            assert set(loaded.files) == required_keys
            artifacts[split] = {name: loaded[name].copy() for name in loaded.files}
        validate_customer_arrays(artifacts[split]["X"], artifacts[split]["y"])
        weeks = artifacts[split]["target_week"]
        assert np.all(weeks[1:] >= weeks[:-1])
        artifact_hashes[split] = actual_hash

    assert artifacts["train"]["target_week"][-1] < artifacts["val"]["target_week"][0]
    assert artifacts["val"]["target_week"][-1] < artifacts["test"]["target_week"][0]

    rows = []
    for split, data in artifacts.items():
        counts = np.bincount(data["y"].astype(int), minlength=2)
        rows.append(
            {
                "split": split.upper(),
                "X shape": str(data["X"].shape),
                "class 0": int(counts[0]),
                "class 1": int(counts[1]),
                "% positive": 100 * counts[1] / counts.sum(),
                "tuần đầu": str(data["target_week"][0]),
                "tuần cuối": str(data["target_week"][-1]),
            }
        )
    pd.DataFrame(rows).set_index("split").round({"% positive": 2})

## 3. Mất cân bằng lớp và class weight

Chỉ khoảng 5–7% mẫu thuộc lớp mua hàng. Accuracy vì thế có thể cao ngay cả khi mô hình gần như luôn dự đoán lớp 0; Precision, Recall, F1 và ROC-AUC phải được xem cùng nhau.

PyTorch đã dùng `positive_weight = n_negative / n_positive`, tính **chỉ từ TRAIN**. Keras dùng cơ chế tương đương qua `class_weight={0: 1, 1: positive_weight}` trong `fit()`. Validation và TEST không tham gia tính trọng số.

In [67]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    class_weight = compute_class_weight(artifacts["train"]["y"])
    expected_positive_weight = customer_meta["class_weight"]["positive_weight"]
    pytorch_positive_weight = pytorch_metrics["class_weighting"]["positive_weight"]
    assert np.isclose(class_weight[1], expected_positive_weight)
    assert np.isclose(class_weight[1], pytorch_positive_weight)

    train_counts = np.bincount(artifacts["train"]["y"].astype(int), minlength=2)
    print(f"TRAIN class 0: {train_counts[0]:,}")
    print(f"TRAIN class 1: {train_counts[1]:,}")
    print(f"class_weight = {class_weight}")

## 4. Keras `SimpleRNN` và sự tương ứng với PyTorch

Hai baseline có cùng ý tưởng:

| PyTorch | Keras | Vai trò |
|---|---|---|
| `nn.RNN(input_size=5, hidden_size=64, batch_first=True)` | `SimpleRNN(64, input_shape=(8, 5))` | Đọc 8 time steps và tạo hidden state cuối |
| `Linear(64, 1)` | `Dense(1, activation="sigmoid")` | Biến hidden state thành đầu ra nhị phân |
| một logit + `BCEWithLogitsLoss` | xác suất sigmoid + `binary_crossentropy` | Tối ưu binary cross-entropy theo cách ổn định của từng framework |

Input Keras có shape `(batch_size, 8, 5)`. `SimpleRNN(64)` trả về hidden state cuối có 64 phần tử cho mỗi sequence vì mặc định `return_sequences=False`. `Dense` và sigmoid chuyển nó thành xác suất mua hàng trong `[0, 1]`.

Số tham số không hoàn toàn bằng PyTorch vì hai framework biểu diễn bias trong RNN khác nhau; đây không phải thay đổi về vai trò của baseline.

In [68]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    HIDDEN_UNITS = 64
    LEARNING_RATE = 1e-3
    model = build_customer_rnn(
        input_shape=(8, 5),
        hidden_units=HIDDEN_UNITS,
        learning_rate=LEARNING_RATE,
    )
    parameter_count = model.count_params()
    model.summary()

    demo_output = model(artifacts["train"]["X"][:4], training=False).numpy().ravel()
    print("Demo input:", artifacts["train"]["X"][:4].shape, "-> probability shape:", demo_output.shape)
    print("Trainable parameters:", f"{parameter_count:,}")
    assert parameter_count == 4545

## 5. `fit()`, binary cross-entropy và EarlyStopping

`model.fit()` lặp qua mini-batch TRAIN, tính binary cross-entropy và dùng Adam để cập nhật trọng số. `validation_data` chỉ đo `val_loss`; nó không được dùng để cập nhật gradient. Để tương ứng với PyTorch, validation loss dùng lại đúng hai trọng số lớp đã tính từ TRAIN (không tính lại từ validation).

`EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)` dừng khi validation loss không còn cải thiện đủ lâu và khôi phục trọng số tốt nhất. `ModelCheckpoint` đồng thời lưu mô hình có validation loss thấp nhất dưới định dạng `.keras`. TEST chưa được dùng ở giai đoạn này.

In [69]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    BATCH_SIZE = 1024
    MAX_EPOCHS = 30
    PATIENCE = 5
    MIN_DELTA = 1e-4

    callbacks = build_callbacks(MODEL_PATH, patience=PATIENCE, min_delta=MIN_DELTA)
    validation_sample_weight = make_sample_weights(artifacts["val"]["y"], class_weight)
    start_time = time.perf_counter()
    history = model.fit(
        artifacts["train"]["X"],
        artifacts["train"]["y"],
        validation_data=(
            artifacts["val"]["X"],
            artifacts["val"]["y"],
            validation_sample_weight,
        ),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=2,
    )
    training_duration_seconds = time.perf_counter() - start_time
    epochs_actually_run = len(history.history["loss"])
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    best_validation_loss = float(np.min(history.history["val_loss"]))

    # ModelCheckpoint lưu raw minimum val_loss; nạp lại trước lần đánh giá TEST duy nhất.
    model = keras.models.load_model(MODEL_PATH)

    print(
        f"Đã chạy {epochs_actually_run} epoch; best epoch={best_epoch}; "
        f"best val_loss={best_validation_loss:.6f}; thời gian={training_duration_seconds:.2f}s"
    )

## 6. Train/validation loss

Loss dưới đây là binary cross-entropy có class weighting. TRAIN loss và validation loss đều phản ánh mục tiêu có trọng số, nhưng chỉ validation loss quyết định checkpoint. Đường loss dùng để chẩn đoán quá trình học, không phải metric cuối cùng của classification.

In [70]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    epochs = np.arange(1, epochs_actually_run + 1)
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    ax.plot(epochs, history.history["loss"], marker="o", label="TRAIN loss")
    ax.plot(epochs, history.history["val_loss"], marker="o", label="Validation loss")
    ax.axvline(best_epoch, color="black", linestyle="--", alpha=0.65, label="Best epoch")
    ax.set(title="Keras Customer RNN — learning curve", xlabel="Epoch", ylabel="Weighted binary cross-entropy")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    loss_figure_path = FIGURE_DIR / "training_validation_loss.png"
    fig.savefig(loss_figure_path, dpi=160, bbox_inches="tight")
    plt.show()

## 7. `predict()` và đánh giá TEST một lần

Sau khi validation đã chọn mô hình, `predict()` tạo một xác suất cho mỗi sequence TEST. Threshold được cố định ở `0.5`, giống PyTorch; threshold không được điều chỉnh từ TEST.

- **Accuracy:** tỷ lệ dự đoán đúng, nhưng dễ gây hiểu nhầm khi lớp 0 áp đảo.
- **Precision:** trong các mẫu dự đoán mua, bao nhiêu mẫu thực sự mua.
- **Recall:** trong các mẫu thực sự mua, mô hình tìm được bao nhiêu.
- **F1-score:** trung bình điều hòa giữa Precision và Recall.
- **ROC-AUC:** khả năng xếp xác suất của mẫu dương cao hơn mẫu âm trên mọi threshold.

In [71]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    THRESHOLD = 0.5
    test_probability = model.predict(
        artifacts["test"]["X"], batch_size=BATCH_SIZE, verbose=0
    ).ravel()
    test_true = artifacts["test"]["y"].astype(np.int8)
    test_predicted = (test_probability >= THRESHOLD).astype(np.int8)
    test_metrics = binary_classification_metrics(
        test_true, test_probability, threshold=THRESHOLD
    )
    test_confusion = confusion_matrix(test_true, test_predicted, labels=[0, 1])

    metric_table = pd.DataFrame(
        {"Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
         "Value": [test_metrics["accuracy"], test_metrics["precision"], test_metrics["recall"], test_metrics["f1"], test_metrics["roc_auc"]]}
    ).set_index("Metric")
    metric_table.round(4)

## 8. Confusion matrix

Ma trận nhầm lẫn cho biết riêng TN, FP, FN và TP tại threshold 0,5. Với dữ liệu mất cân bằng, đây là cách nhìn trực tiếp hơn so với chỉ báo cáo Accuracy.

In [72]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    fig, ax = plt.subplots(figsize=(5.6, 5.0))
    display = ConfusionMatrixDisplay(test_confusion, display_labels=["Không mua (0)", "Mua (1)"])
    display.plot(ax=ax, cmap="Blues", colorbar=False, values_format=",d")
    ax.set_title("Keras Customer RNN — TEST confusion matrix")
    fig.tight_layout()
    confusion_figure_path = FIGURE_DIR / "test_confusion_matrix.png"
    fig.savefig(confusion_figure_path, dpi=160, bbox_inches="tight")
    plt.show()

## 9. ROC curve

ROC curve quét qua nhiều threshold và biểu diễn đánh đổi giữa true-positive rate và false-positive rate. Đường chéo là mức xếp hạng ngẫu nhiên; AUC tóm tắt diện tích dưới đường cong. ROC-AUC không thay thế Precision/Recall tại threshold triển khai.

In [73]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    false_positive_rate, true_positive_rate, _ = roc_curve(test_true, test_probability)
    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    ax.plot(false_positive_rate, true_positive_rate, linewidth=2, label=f"Keras RNN (AUC={test_metrics['roc_auc']:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Ngẫu nhiên")
    ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="Keras Customer RNN — TEST ROC curve")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.01)
    ax.grid(alpha=0.25)
    ax.legend(loc="lower right")
    fig.tight_layout()
    roc_figure_path = FIGURE_DIR / "test_roc_curve.png"
    fig.savefig(roc_figure_path, dpi=160, bbox_inches="tight")
    plt.show()

## 10. Lưu predictions và metadata thí nghiệm

Mỗi prediction giữ nguyên `CustomerID` và `target_week` để truy vết, nhưng hai trường này không phải feature. JSON sử dụng cấu trúc gần với artifact PyTorch: kiến trúc, dữ liệu, huấn luyện, class weighting, đánh giá và đường dẫn artifact.

In [74]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    predictions = pd.DataFrame(
        {
            "CustomerID": artifacts["test"]["customer_id"],
            "target_week": artifacts["test"]["target_week"].astype(str),
            "true_label": test_true,
            "probability": test_probability,
            "predicted_label": test_predicted,
        }
    )
    assert not predictions.isna().any().any()
    assert not predictions.duplicated(["CustomerID", "target_week"]).any()
    predictions.to_csv(PREDICTIONS_PATH, index=False)

    model_hash = sha256(MODEL_PATH)
    predictions_hash = sha256(PREDICTIONS_PATH)
    metrics_payload = {
        "experiment": "Keras Customer Vanilla RNN",
        "seed": SEED,
        "device": device_label.lower(),
        "framework_versions": {"tensorflow": tf.__version__, "keras": keras.__version__},
        "architecture": {
            "recurrent_layer": "keras.layers.SimpleRNN",
            "input_size": 5,
            "sequence_length": 8,
            "hidden_units": HIDDEN_UNITS,
            "num_layers": 1,
            "activation": "tanh",
            "head": "Dense(1, activation='sigmoid') producing one probability",
            "parameter_count": parameter_count,
        },
        "data": {
            "feature_names": customer_meta["feature_names"],
            "split_shapes": {split: list(data["X"].shape) for split, data in artifacts.items()},
            "source_artifacts": customer_meta["artifacts"],
        },
        "training": {
            "optimizer": "Adam",
            "learning_rate": LEARNING_RATE,
            "criterion": "binary_crossentropy",
            "batch_size": BATCH_SIZE,
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "min_delta": MIN_DELTA,
            "restore_best_weights": True,
            "validation_weighting": "sample weights mapped from TRAIN-derived class_weight",
            "epochs_actually_run": epochs_actually_run,
            "best_epoch": best_epoch,
            "best_validation_loss": best_validation_loss,
            "training_duration_seconds": training_duration_seconds,
            "selection_scope": "validation loss only; TEST unused until final evaluation",
            "history": {
                "train_loss": [float(value) for value in history.history["loss"]],
                "validation_loss": [float(value) for value in history.history["val_loss"]],
            },
        },
        "class_weighting": {
            "used": True,
            "type": "Keras class_weight passed to fit()",
            "class_0_weight": class_weight[0],
            "positive_weight": class_weight[1],
            "formula": "TRAIN negatives / TRAIN positives",
            "fit_scope": "TRAIN y only",
            "train_class_0": int(train_counts[0]),
            "train_class_1": int(train_counts[1]),
        },
        "evaluation": {
            "split": "TEST",
            "evaluations_on_test": 1,
            "threshold": THRESHOLD,
            "metrics": test_metrics,
            "confusion_matrix_tn_fp_fn_tp": {
                "tn": int(test_confusion[0, 0]),
                "fp": int(test_confusion[0, 1]),
                "fn": int(test_confusion[1, 0]),
                "tp": int(test_confusion[1, 1]),
            },
        },
        "artifacts": {
            "model": {"path": str(MODEL_PATH.relative_to(ROOT)).replace("\\", "/"), "sha256": model_hash},
            "predictions": {"path": str(PREDICTIONS_PATH.relative_to(ROOT)).replace("\\", "/"), "rows": len(predictions), "sha256": predictions_hash},
            "figures": [
                str(path.relative_to(ROOT)).replace("\\", "/")
                for path in (loss_figure_path, confusion_figure_path, roc_figure_path)
            ],
        },
    }
    METRICS_PATH.write_text(json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"Đã lưu model -> {MODEL_PATH.relative_to(ROOT)}")
    print(f"Đã lưu {len(predictions):,} predictions -> {PREDICTIONS_PATH.relative_to(ROOT)}")
    print(f"Đã lưu metrics -> {METRICS_PATH.relative_to(ROOT)}")
    predictions.head()

## 11. Kiểm tra artifact đầu ra

Kiểm tra cuối xác nhận model có thể nạp lại, CSV có đúng schema và số dòng, xác suất hợp lệ, metrics JSON đầy đủ, ba figure tồn tại, và checksum input vẫn nguyên vẹn.

In [75]:
if RUN_EXPERIMENTS and ACTIVE_FRAMEWORK == 'keras':
    expected_columns = ["CustomerID", "target_week", "true_label", "probability", "predicted_label"]
    assert MODEL_PATH.is_file()
    assert METRICS_PATH.is_file()
    assert PREDICTIONS_PATH.is_file()
    assert predictions.columns.tolist() == expected_columns
    assert len(predictions) == len(artifacts["test"]["y"]) == 53052
    assert predictions["probability"].between(0, 1).all()
    assert set(predictions["predicted_label"].unique()).issubset({0, 1})
    assert all(path.is_file() for path in (loss_figure_path, confusion_figure_path, roc_figure_path))

    reloaded = keras.models.load_model(MODEL_PATH)
    assert reloaded.input_shape == (None, 8, 5)
    assert reloaded.output_shape == (None, 1)
    assert reloaded.count_params() == parameter_count

    saved_metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
    assert saved_metrics["evaluation"]["threshold"] == 0.5
    assert saved_metrics["evaluation"]["evaluations_on_test"] == 1
    for name in ("accuracy", "precision", "recall", "f1", "roc_auc"):
        assert np.isfinite(saved_metrics["evaluation"]["metrics"][name])
    for split, expected_hash in artifact_hashes.items():
        input_path = ROOT / customer_meta["artifacts"][split]["path"]
        assert sha256(input_path) == expected_hash

    print("PASS — notebook chạy đầy đủ, model/CSV/JSON/figures hợp lệ, input không đổi.")
    print("Figures:", sorted(path.name for path in FIGURE_DIR.glob("*.png")))

## 12. Tóm tắt

- Keras nhận cùng tensor `(samples, 8, 5)`, cùng target tuần kế tiếp và cùng split thời gian với PyTorch.
- `SimpleRNN(64)` đóng vai trò tương ứng `nn.RNN`; `Dense(1, sigmoid)` tương ứng `Linear` rồi chuyển logit thành xác suất.
- Binary cross-entropy là loss huấn luyện; class weight dương được tính duy nhất từ TRAIN.
- `fit()` dùng validation cho EarlyStopping và khôi phục mô hình tốt nhất; TEST chỉ được dự đoán một lần sau lựa chọn mô hình.
- Accuracy, Precision, Recall, F1, ROC-AUC, confusion matrix và ROC curve cùng mô tả kết quả trên dữ liệu mất cân bằng.
- Notebook này ghi kết quả Keras độc lập. Việc so sánh và kết luận giữa framework được dành cho Notebook 08.

## Kết quả gốc để đối chiếu

Các file đã lưu được đọc nguyên trạng; không fit hoặc forward TEST mới.

In [76]:
import json
import pandas as pd
from IPython.display import display
for _relative in ['A06/results/metrics/keras_customer_metrics.json', 'A06/results/metrics/keras_stock_metrics.json', 'A06/results/metrics/pytorch_customer_metrics.json', 'A06/results/metrics/pytorch_stock_metrics.json']:
    _path = REPO / _relative
    print(_relative)
    if _path.suffix == ".csv": display(pd.read_csv(_path))
    else: display(json.loads(_path.read_text()))


A06/results/metrics/keras_customer_metrics.json


{'experiment': 'Keras Customer Vanilla RNN',
 'seed': 42,
 'device': 'cpu',
 'framework_versions': {'tensorflow': '2.21.0', 'keras': '3.15.1'},
 'architecture': {'recurrent_layer': 'keras.layers.SimpleRNN',
  'input_size': 5,
  'sequence_length': 8,
  'hidden_units': 64,
  'num_layers': 1,
  'activation': 'tanh',
  'head': "Dense(1, activation='sigmoid') producing one probability",
  'parameter_count': 4545},
 'data': {'feature_names': ['total_spent',
   'total_quantity',
   'order_count',
   'unique_products',
   'active_flag'],
  'split_shapes': {'train': [247458, 8, 5],
   'val': [50354, 8, 5],
   'test': [53052, 8, 5]},
  'source_artifacts': {'test': {'bytes': 418292,
    'path': 'datasets/processed/customer_test.npz',
    'sha256': 'de8cbcec15e1ede07f374573b33bfc1c31816191fc69d83e9a1bfc062b0b0bef'},
   'train': {'bytes': 1976810,
    'path': 'datasets/processed/customer_train.npz',
    'sha256': '1f3995f0b2c4295fd824ab1fba35ff6208dc954ceb2d091dc979fd4d71085051'},
   'val': {'bytes

A06/results/metrics/keras_stock_metrics.json


{'task': "30 previous trading days x 5 features -> next trading day's Close",
 'framework': 'Keras',
 'framework_versions': {'tensorflow': '2.21.0', 'keras': '3.15.1'},
 'architecture': {'recurrent_layer': 'keras.layers.SimpleRNN',
  'input_size': 5,
  'sequence_length': 30,
  'hidden_units': 64,
  'num_layers': 1,
  'activation': 'tanh',
  'head': "Dense(1, activation='linear')"},
 'parameter_count': 4545,
 'seed': 42,
 'batch_size': 64,
 'optimizer': 'Adam',
 'learning_rate': 0.001,
 'loss': 'MSE on scaled target',
 'max_epochs': 30,
 'patience': 5,
 'min_delta': 0.0001,
 'restore_best_weights': True,
 'epochs': 30,
 'epochs_actually_run': 30,
 'best_epoch': 30,
 'best_validation_loss': 0.004865891300141811,
 'best_validation_loss_space': 'standardized target',
 'device': 'cpu',
 'training_duration_seconds': 21.468527700000095,
 'selection_scope': 'validation loss only; TEST unused until final evaluation',
 'training_history': {'train_loss': [0.06277468055486679,
   0.008362506516277

A06/results/metrics/pytorch_customer_metrics.json


{'experiment': 'PyTorch Customer Vanilla RNN',
 'seed': 42,
 'device': 'cpu',
 'architecture': {'recurrent_layer': 'torch.nn.RNN',
  'input_size': 5,
  'sequence_length': 8,
  'hidden_size': 64,
  'num_layers': 1,
  'batch_first': True,
  'nonlinearity': 'tanh',
  'head': 'Linear(64, 1) producing one logit',
  'sigmoid_inside_model': False,
  'parameter_count': 4609},
 'data': {'feature_names': ['total_spent',
   'total_quantity',
   'order_count',
   'unique_products',
   'active_flag'],
  'split_shapes': {'train': [247458, 8, 5],
   'val': [50354, 8, 5],
   'test': [53052, 8, 5]},
  'source_artifacts': {'test': {'bytes': 418292,
    'path': 'datasets/processed/customer_test.npz',
    'sha256': 'de8cbcec15e1ede07f374573b33bfc1c31816191fc69d83e9a1bfc062b0b0bef'},
   'train': {'bytes': 1976810,
    'path': 'datasets/processed/customer_train.npz',
    'sha256': '1f3995f0b2c4295fd824ab1fba35ff6208dc954ceb2d091dc979fd4d71085051'},
   'val': {'bytes': 343645,
    'path': 'datasets/processed

A06/results/metrics/pytorch_stock_metrics.json


{'task': "30 previous trading days x 5 features -> next trading day's Close",
 'framework': 'PyTorch',
 'architecture': {'recurrent_layer': 'torch.nn.RNN',
  'input_size': 5,
  'hidden_size': 64,
  'num_layers': 1,
  'batch_first': True,
  'nonlinearity': 'tanh',
  'head': 'Linear(64, 1)'},
 'parameter_count': 4609,
 'seed': 42,
 'batch_size': 64,
 'optimizer': 'Adam',
 'learning_rate': 0.001,
 'loss': 'MSELoss on scaled target',
 'max_epochs': 30,
 'patience': 5,
 'epochs': 14,
 'epochs_actually_run': 14,
 'best_epoch': 9,
 'best_validation_loss': 0.0065248434345743225,
 'best_validation_loss_space': 'standardized target',
 'device': 'cpu',
 'training_duration_seconds': 5.613410699999974,
 'MAE': 20.339256857274822,
 'RMSE': 23.57881412768672,
 'R2': -0.026903235017200933,
 'naive_MAE': 2.6177071920255335,
 'naive_RMSE': 3.8789281161189058,
 'naive_R2': 0.9722086869082393,
 'test_metrics_real_scale': {'mae': 20.339256857274822,
  'rmse': 23.57881412768672,
  'r2': -0.02690323501720093

In [77]:
import inspect
_notebook_defined = [(name, key) for name, mod in INLINE_MODULES.items() for key, obj in vars(mod).items() if inspect.isfunction(obj) and obj.__module__ == name]
assert all(vars(INLINE_MODULES[name])[key].__code__.co_filename.startswith("<notebook>/") for name,key in _notebook_defined)
print({"module có code ngay trong notebook": sum(bool(getattr(m,"__notebook_source__",False)) for m in INLINE_MODULES.values()), "hàm inline":len(_notebook_defined), "train/TEST đã chạy":False})


{'module có code ngay trong notebook': 4, 'hàm inline': 24, 'train/TEST đã chạy': False}
